<a href="https://colab.research.google.com/github/Praseedas/example-repo/blob/main/Variant_Annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/Task_3_Variant_Annotation"

folders = [
    "data/raw",
    "data/processed",
    "results/annotated",
    "results/tables",
    "reports",
    "scripts"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Task 3 workspace created successfully!")

Mounted at /content/drive
Task 3 workspace created successfully!


In [3]:
import os
import requests

url = "https://raw.githubusercontent.com/cBioPortal/datahub/master/public/luad_tcga_pan_can_atlas_2018/data_mutations.txt"

output_file = "/content/Task_3_Variant_Annotation/data/raw/data_mutations.txt"

response = requests.get(url)

if response.status_code == 200:
    with open(output_file, "wb") as f:
        f.write(response.content)

    print("Dataset downloaded successfully!")
    print("File size:", round(os.path.getsize(output_file) / (1024*1024), 2), "MB")
else:
    print("Download failed. Status code:", response.status_code)

Dataset downloaded successfully!
File size: 0.0 MB


In [4]:
import os

file_path = "/content/Task_3_Variant_Annotation/data/raw/data_mutations.txt"

print("File exists:", os.path.exists(file_path))
print("Exact file size:", os.path.getsize(file_path), "bytes")

with open(file_path, "r") as f:
    for i in range(10):
        line = f.readline()
        if not line:
            break
        print(line.rstrip())

File exists: True
Exact file size: 134 bytes
version https://git-lfs.github.com/spec/v1
oid sha256:a08847f9536886be6de4399caabf87202afacf19ee79f0500284020d58827079
size 232136490


In [6]:

import requests

base_url = "https://api.gdc.cancer.gov"

# Check GDC API connection
response = requests.get(f"{base_url}/status", timeout=30)

print("GDC API status:", response.status_code)

if response.status_code == 200:
    print("Connected successfully!")

# Verify both TCGA projects
for project in ["TCGA-LUAD", "TCGA-LUSC"]:
    r = requests.get(
        f"{base_url}/projects/{project}",
        timeout=30
    )

    if r.status_code == 200:
        data = r.json()["data"]
        print("\nProject:", data["project_id"])
        print("Name:", data["name"])
        print("Primary site:", data.get("primary_site"))
    else:
        print("\nCould not retrieve:", project, r.status_code)


GDC API status: 200
Connected successfully!

Project: TCGA-LUAD
Name: Lung Adenocarcinoma
Primary site: ['Bronchus and lung']

Project: TCGA-LUSC
Name: Lung Squamous Cell Carcinoma
Primary site: ['Bronchus and lung']


In [7]:

import requests
import json
import os

base_url = "https://api.gdc.cancer.gov/files"

filters = {
    "op": "and",
    "content": [
        {
            "op": "in",
            "content": {
                "field": "cases.project.project_id",
                "value": ["TCGA-LUAD", "TCGA-LUSC"]
            }
        },
        {
            "op": "=",
            "content": {
                "field": "data_category",
                "value": "Simple Nucleotide Variation"
            }
        },
        {
            "op": "=",
            "content": {
                "field": "data_type",
                "value": "Masked Somatic Mutation"
            }
        },
        {
            "op": "=",
            "content": {
                "field": "data_format",
                "value": "MAF"
            }
        },
        {
            "op": "=",
            "content": {
                "field": "access",
                "value": "open"
            }
        }
    ]
}

params = {
    "filters": json.dumps(filters),
    "fields": "file_id,file_name,file_size,data_format,data_type,access,cases.project.project_id",
    "format": "JSON",
    "size": 100
}

response = requests.get(base_url, params=params, timeout=60)

print("API status:", response.status_code)

if response.status_code == 200:
    results = response.json()["data"]

    print("Total matching files:", results["pagination"]["total"])

    for file in results["hits"]:
        projects = sorted(set(
            case["project"]["project_id"]
            for case in file.get("cases", [])
            if case.get("project")
        ))

        print("\nFile:", file["file_name"])
        print("Projects:", projects)
        print("Size (MB):", round(file["file_size"] / (1024 * 1024), 2))
        print("File ID:", file["file_id"])
else:
    print(response.text[:2000])


API status: 200
Total matching files: 1167

File: 14826ad4-fe64-4f31-bb3e-8e2d57ad7f5f.wxs.aliquot_ensemble_masked.maf.gz
Projects: ['TCGA-LUAD']
Size (MB): 0.23
File ID: 374e13e1-98a8-4b65-94da-0d78ae51df5a

File: 455d27a7-9bca-4f47-86ce-38afccfe9ab5.wxs.aliquot_ensemble_masked.maf.gz
Projects: ['TCGA-LUAD']
Size (MB): 0.04
File ID: 8416388e-8a30-441d-b14b-bfcec4daf78a

File: f099efdd-6021-4afd-9748-733e5056c0a4.wxs.aliquot_ensemble_masked.maf.gz
Projects: ['TCGA-LUAD']
Size (MB): 0.05
File ID: 683d979f-0ced-4839-b61b-c26f065c1da6

File: 1a47b275-62bc-42a1-8543-65caaacf5ed3.wxs.aliquot_ensemble_masked.maf.gz
Projects: ['TCGA-LUAD']
Size (MB): 0.07
File ID: 101c7666-42ea-45d9-8288-16a3cbd47274

File: f4899380-8783-4cdb-93fb-862e8f9ff402.wxs.aliquot_ensemble_masked.maf.gz
Projects: ['TCGA-LUAD']
Size (MB): 0.05
File ID: fde996d6-190e-4084-9b51-ab8b9e9ceea6

File: f2cfacbc-a824-4568-89db-6aca0d06af74.wxs.aliquot_ensemble_masked.maf.gz
Projects: ['TCGA-LUAD']
Size (MB): 0.03
File ID: c791

In [8]:

import requests
import json

base_url = "https://api.gdc.cancer.gov/files"

for project in ["TCGA-LUAD", "TCGA-LUSC"]:

    filters = {
        "op": "and",
        "content": [
            {
                "op": "=",
                "content": {
                    "field": "cases.project.project_id",
                    "value": project
                }
            },
            {
                "op": "=",
                "content": {
                    "field": "data_category",
                    "value": "Simple Nucleotide Variation"
                }
            },
            {
                "op": "=",
                "content": {
                    "field": "data_type",
                    "value": "Masked Somatic Mutation"
                }
            },
            {
                "op": "=",
                "content": {
                    "field": "data_format",
                    "value": "MAF"
                }
            },
            {
                "op": "=",
                "content": {
                    "field": "access",
                    "value": "open"
                }
            }
        ]
    }

    params = {
        "filters": json.dumps(filters),
        "fields": "file_id,file_name,file_size,cases.submitter_id",
        "format": "JSON",
        "size": 5,
        "sort": "file_size:desc"
    }

    response = requests.get(base_url, params=params, timeout=60)
    response.raise_for_status()

    data = response.json()["data"]

    print("\n" + "=" * 55)
    print("PROJECT:", project)
    print("TOTAL AVAILABLE FILES:", data["pagination"]["total"])

    for file in data["hits"]:
        print("\nFile:", file["file_name"])
        print("Size (MB):", round(file["file_size"] / (1024 * 1024), 3))
        print("File ID:", file["file_id"])
        print("Cases:", [
            case.get("submitter_id")
            for case in file.get("cases", [])
        ])



PROJECT: TCGA-LUAD
TOTAL AVAILABLE FILES: 618

File: 18aded85-5a36-47dd-b6aa-04161318df0c.wxs.aliquot_ensemble_masked.maf.gz
Size (MB): 0.643
File ID: d7718619-286f-47ec-9b8c-430fb1a01383
Cases: ['TCGA-17-Z031']

File: 3cfdf0dc-e83a-4001-87d6-89d570936fcc.wxs.aliquot_ensemble_masked.maf.gz
Size (MB): 0.554
File ID: ad708c2d-e1b7-4e08-be32-b71c26310c41
Cases: ['TCGA-55-8506']

File: d30c9651-0eee-4446-aa6f-5dbbb1c3cab4.wxs.aliquot_ensemble_masked.maf.gz
Size (MB): 0.498
File ID: 879297cc-5afe-48aa-9a38-e8b1c6fe0a32
Cases: ['TCGA-05-4382']

File: e6954da3-287b-4a5f-b4de-e690fb8ed425.wxs.aliquot_ensemble_masked.maf.gz
Size (MB): 0.479
File ID: 44a776b7-0eb2-463c-ab1d-80c654278f56
Cases: ['TCGA-L9-A7SV']

File: 778cde8c-6445-4b59-9185-3cc50f51836d.wxs.aliquot_ensemble_masked.maf.gz
Size (MB): 0.467
File ID: 03add666-b87c-414c-8695-5667f3b599c1
Cases: ['TCGA-55-7994']

PROJECT: TCGA-LUSC
TOTAL AVAILABLE FILES: 549

File: 18e45dec-54d0-4600-b636-47781d9842c1.wxs.aliquot_ensemble_masked.maf.

In [9]:

import requests
import gzip
import os
import pandas as pd

# Folder for raw GDC files
raw_dir = "/content/Task_3_Variant_Annotation/data/raw"
os.makedirs(raw_dir, exist_ok=True)

# Five selected TCGA-LUAD files
files = {
    "TCGA-17-Z031": "d7718619-286f-47ec-9b8c-430fb1a01383",
    "TCGA-55-8506": "ad708c2d-e1b7-4e08-be32-b71c26310c41",
    "TCGA-05-4382": "879297cc-5afe-48aa-9a38-e8b1c6fe0a32",
    "TCGA-L9-A7SV": "44a776b7-0eb2-463c-ab1d-80c654278f56",
    "TCGA-55-7994": "03add666-b87c-414c-8695-5667f3b599c1"
}

base_url = "https://api.gdc.cancer.gov/data/"

for sample, file_id in files.items():
    output_path = os.path.join(raw_dir, f"{sample}.maf.gz")

    print(f"\nDownloading {sample}...")

    response = requests.get(base_url + file_id, stream=True, timeout=180)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    print("Saved:", output_path)
    print("File size (MB):", round(os.path.getsize(output_path) / (1024 * 1024), 3))

print("\nAll five LUAD MAF files downloaded successfully.")



Saved: /content/Task_3_Variant_Annotation/data/raw/TCGA-17-Z031.maf.gz
File size (MB): 0.643

Saved: /content/Task_3_Variant_Annotation/data/raw/TCGA-55-8506.maf.gz
File size (MB): 0.554

Saved: /content/Task_3_Variant_Annotation/data/raw/TCGA-05-4382.maf.gz
File size (MB): 0.498

Saved: /content/Task_3_Variant_Annotation/data/raw/TCGA-L9-A7SV.maf.gz
File size (MB): 0.479

Saved: /content/Task_3_Variant_Annotation/data/raw/TCGA-55-7994.maf.gz
File size (MB): 0.467

All five LUAD MAF files downloaded successfully.


In [10]:

import os
import gzip
import pandas as pd

raw_dir = "/content/Task_3_Variant_Annotation/data/raw"

for filename in sorted(os.listdir(raw_dir)):
    if filename.endswith(".maf.gz"):
        filepath = os.path.join(raw_dir, filename)

        print("\n" + "=" * 65)
        print("SAMPLE:", filename)

        # Read MAF while skipping metadata/comment lines
        with gzip.open(filepath, "rt") as f:
            lines = [line for line in f if not line.startswith("#")]

        from io import StringIO
        maf = pd.read_csv(StringIO("".join(lines)), sep="\t", low_memory=False)

        print("Total variant records:", len(maf))
        print("Number of columns:", len(maf.columns))
        print("Genome build:", maf["NCBI_Build"].dropna().unique()
              if "NCBI_Build" in maf.columns else "Not specified")

        print("\nFirst 10 column names:")
        print(maf.columns.tolist()[:10])

        print("\nFirst 3 variant records:")
        display(maf.head(3))



SAMPLE: TCGA-05-4382.maf.gz
Total variant records: 1824
Number of columns: 140
Genome build: ['GRCh38']

First 10 column names:
['Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Strand', 'Variant_Classification', 'Variant_Type']

First 3 variant records:


,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Variant_Classification,Variant_Type,...,normal_bam_uuid,case_id,GDC_FILTER,COSMIC,hotspot,RNA_Support,RNA_depth,RNA_ref_count,RNA_alt_count,callers
0,UBE2J2,118424,BI,GRCh38,chr1,1255389,1255389,+,Silent,SNP,...,f0be1c9e-728c-4655-b792-96bfe1cd5344,3434b91a-c05f-460f-a078-7b1bb6e7085d,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2
1,CPTP,80772,BI,GRCh38,chr1,1326974,1326974,+,Missense_Mutation,SNP,...,f0be1c9e-728c-4655-b792-96bfe1cd5344,3434b91a-c05f-460f-a078-7b1bb6e7085d,NaN,NaN,N,Unknown,NaN,NaN,NaN,mutect2;varscan2
2,CFAP74,85452,BI,GRCh38,chr1,1988951,1988951,+,Silent,SNP,...,f0be1c9e-728c-4655-b792-96bfe1cd5344,3434b91a-c05f-460f-a078-7b1bb6e7085d,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2



SAMPLE: TCGA-17-Z031.maf.gz
Total variant records: 2468
Number of columns: 140
Genome build: ['GRCh38']

First 10 column names:
['Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Strand', 'Variant_Classification', 'Variant_Type']

First 3 variant records:


,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Variant_Classification,Variant_Type,...,normal_bam_uuid,case_id,GDC_FILTER,COSMIC,hotspot,RNA_Support,RNA_depth,RNA_ref_count,RNA_alt_count,callers
0,TNFRSF4,7293,BI,GRCh38,chr1,1212125,1212125,+,Missense_Mutation,SNP,...,bb97a9eb-2775-40c1-aecf-aa7e5b83a8f0,c43c8eea-7014-4abe-b805-397a9062d3e0,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2
1,CFAP74,85452,BI,GRCh38,chr1,1988600,1988600,+,Missense_Mutation,SNP,...,bb97a9eb-2775-40c1-aecf-aa7e5b83a8f0,c43c8eea-7014-4abe-b805-397a9062d3e0,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2
2,CFAP74,85452,BI,GRCh38,chr1,1988944,1988944,+,Missense_Mutation,SNP,...,bb97a9eb-2775-40c1-aecf-aa7e5b83a8f0,c43c8eea-7014-4abe-b805-397a9062d3e0,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2



SAMPLE: TCGA-55-7994.maf.gz
Total variant records: 1704
Number of columns: 140
Genome build: ['GRCh38']

First 10 column names:
['Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Strand', 'Variant_Classification', 'Variant_Type']

First 3 variant records:


,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Variant_Classification,Variant_Type,...,normal_bam_uuid,case_id,GDC_FILTER,COSMIC,hotspot,RNA_Support,RNA_depth,RNA_ref_count,RNA_alt_count,callers
0,KLHL17,339451,BI,GRCh38,chr1,962470,962470,+,Missense_Mutation,SNP,...,8a26dec8-750d-4964-8874-d02647a449f0,67dbe286-edb3-4d07-8f74-9e5254f01945,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2
1,ATAD3B,83858,BI,GRCh38,chr1,1490391,1490391,+,3'UTR,SNP,...,8a26dec8-750d-4964-8874-d02647a449f0,67dbe286-edb3-4d07-8f74-9e5254f01945,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2
2,PRDM16,63976,BI,GRCh38,chr1,3430894,3430894,+,Missense_Mutation,SNP,...,8a26dec8-750d-4964-8874-d02647a449f0,67dbe286-edb3-4d07-8f74-9e5254f01945,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2



SAMPLE: TCGA-55-8506.maf.gz
Total variant records: 1922
Number of columns: 140
Genome build: ['GRCh38']

First 10 column names:
['Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Strand', 'Variant_Classification', 'Variant_Type']

First 3 variant records:


,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Variant_Classification,Variant_Type,...,normal_bam_uuid,case_id,GDC_FILTER,COSMIC,hotspot,RNA_Support,RNA_depth,RNA_ref_count,RNA_alt_count,callers
0,INTS11,54973,BI,GRCh38,chr1,1312839,1312839,+,Silent,SNP,...,aad7c7fc-109b-48b4-9744-719cc29d528f,3ed05070-6633-4de5-9f72-090f93aa2e4d,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2
1,INTS11,54973,BI,GRCh38,chr1,1320494,1320494,+,Silent,SNP,...,aad7c7fc-109b-48b4-9744-719cc29d528f,3ed05070-6633-4de5-9f72-090f93aa2e4d,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2
2,TAS1R3,83756,BI,GRCh38,chr1,1331842,1331842,+,Silent,SNP,...,aad7c7fc-109b-48b4-9744-719cc29d528f,3ed05070-6633-4de5-9f72-090f93aa2e4d,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2



SAMPLE: TCGA-L9-A7SV.maf.gz
Total variant records: 1766
Number of columns: 140
Genome build: ['GRCh38']

First 10 column names:
['Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Strand', 'Variant_Classification', 'Variant_Type']

First 3 variant records:


,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Variant_Classification,Variant_Type,...,normal_bam_uuid,case_id,GDC_FILTER,COSMIC,hotspot,RNA_Support,RNA_depth,RNA_ref_count,RNA_alt_count,callers
0,AGRN,375790,BI,GRCh38,chr1,1047396,1047396,+,Missense_Mutation,SNP,...,15756a48-affe-40bd-a9ed-b218270439a2,54e90433-5218-4d83-848f-f995ecba2a63,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2
1,ACTRT2,140625,BI,GRCh38,chr1,3022045,3022045,+,Missense_Mutation,SNP,...,15756a48-affe-40bd-a9ed-b218270439a2,54e90433-5218-4d83-848f-f995ecba2a63,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2
2,MEGF6,1953,BI,GRCh38,chr1,3595367,3595367,+,Missense_Mutation,SNP,...,15756a48-affe-40bd-a9ed-b218270439a2,54e90433-5218-4d83-848f-f995ecba2a63,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2


In [11]:

import os
import gzip
import pandas as pd
from io import StringIO

raw_dir = "/content/Task_3_Variant_Annotation/data/raw"

# Load all five MAF files
all_mafs = []

for filename in sorted(os.listdir(raw_dir)):
    if filename.endswith(".maf.gz"):
        filepath = os.path.join(raw_dir, filename)

        with gzip.open(filepath, "rt") as f:
            lines = [line for line in f if not line.startswith("#")]

        maf = pd.read_csv(StringIO("".join(lines)), sep="\t",
                          low_memory=False)

        maf["Sample_ID"] = filename.replace(".maf.gz", "")
        all_mafs.append(maf)

combined_maf = pd.concat(all_mafs, ignore_index=True)

print("Combined dataset shape:", combined_maf.shape)

# Find important annotation columns
keywords = [
    "HGV", "Protein", "Transcript", "Allele",
    "Depth", "VAF", "COSMIC", "hotspot",
    "FILTER", "t_depth", "t_alt_count",
    "t_ref_count", "gnomAD", "dbSNP"
]

print("\nRelevant annotation columns:")
for col in combined_maf.columns:
    if any(word.lower() in col.lower() for word in keywords):
        print(col)

print("\nVariant classification counts:")
print(combined_maf["Variant_Classification"].value_counts().head(15))

print("\nVariant type counts:")
print(combined_maf["Variant_Type"].value_counts())


Combined dataset shape: (9684, 141)

Relevant annotation columns:
Reference_Allele
Tumor_Seq_Allele1
Tumor_Seq_Allele2
dbSNP_RS
dbSNP_Val_Status
Match_Norm_Seq_Allele1
Match_Norm_Seq_Allele2
Tumor_Validation_Allele1
Tumor_Validation_Allele2
Match_Norm_Validation_Allele1
Match_Norm_Validation_Allele2
HGVSc
HGVSp
HGVSp_Short
Transcript_ID
t_depth
t_ref_count
t_alt_count
n_depth
Allele
Protein_position
TRANSCRIPT_STRAND
gnomAD_AF
gnomAD_AFR_AF
gnomAD_AMR_AF
gnomAD_ASJ_AF
gnomAD_EAS_AF
gnomAD_FIN_AF
gnomAD_NFE_AF
gnomAD_OTH_AF
gnomAD_SAS_AF
gnomAD_non_cancer_AF
gnomAD_non_cancer_AFR_AF
gnomAD_non_cancer_AMI_AF
gnomAD_non_cancer_AMR_AF
gnomAD_non_cancer_ASJ_AF
gnomAD_non_cancer_EAS_AF
gnomAD_non_cancer_FIN_AF
gnomAD_non_cancer_MID_AF
gnomAD_non_cancer_NFE_AF
gnomAD_non_cancer_OTH_AF
gnomAD_non_cancer_SAS_AF
gnomAD_non_cancer_MAX_AF_adj
gnomAD_non_cancer_MAX_AF_POPS_adj
TRANSCRIPTION_FACTORS
HGVS_OFFSET
GDC_FILTER
COSMIC
hotspot
RNA_depth

Variant classification counts:
Variant_Classificatio

In [12]:

# Functional variant classes to prioritise
important_classes = [
    "Missense_Mutation",
    "Nonsense_Mutation",
    "Frame_Shift_Del",
    "Frame_Shift_Ins",
    "Splice_Site",
    "In_Frame_Del",
    "In_Frame_Ins",
    "Translation_Start_Site"
]

# Lung cancer-associated genes for focused review
lung_genes = [
    "EGFR", "KRAS", "TP53", "STK11", "KEAP1",
    "BRAF", "MET", "ERBB2", "PIK3CA", "RB1", "NF1"
]

# Filter potentially protein-altering variants
candidate_variants = combined_maf[
    combined_maf["Variant_Classification"].isin(important_classes)
].copy()

print("Total potentially protein-altering records:",
      len(candidate_variants))

print("\nVariant classes in candidate dataset:")
print(candidate_variants["Variant_Classification"].value_counts())

# Focus on variants in selected lung cancer-associated genes
driver_candidates = candidate_variants[
    candidate_variants["Hugo_Symbol"].isin(lung_genes)
].copy()

print("\nVariants in selected lung cancer-associated genes:",
      len(driver_candidates))

print("\nGene-wise variant counts:")
print(driver_candidates["Hugo_Symbol"].value_counts())

# Display useful annotation fields
display_columns = [
    "Sample_ID", "Hugo_Symbol", "Chromosome",
    "Start_Position", "Reference_Allele",
    "Tumor_Seq_Allele2", "Variant_Classification",
    "HGVSc", "HGVSp_Short", "gnomAD_AF",
    "COSMIC", "hotspot", "t_depth", "t_alt_count"
]

display_columns = [
    col for col in display_columns
    if col in driver_candidates.columns
]

print("\nPrioritised candidate variants:")
display(
    driver_candidates[display_columns]
    .sort_values(["Hugo_Symbol", "Sample_ID"])
    .head(30)
)

# Check availability of evidence fields
print("\nEvidence field completeness:")
for col in ["HGVSc", "HGVSp_Short", "gnomAD_AF",
            "COSMIC", "hotspot", "t_depth", "t_alt_count"]:
    if col in driver_candidates.columns:
        print(
            col, ":",
            driver_candidates[col].notna().sum(),
            "non-missing records out of",
            len(driver_candidates)
        )


Total potentially protein-altering records: 7284

Variant classes in candidate dataset:
Variant_Classification
Missense_Mutation         6253
Nonsense_Mutation          504
Frame_Shift_Del            238
Splice_Site                181
Frame_Shift_Ins             75
In_Frame_Del                17
Translation_Start_Site      15
In_Frame_Ins                 1
Name: count, dtype: int64

Variants in selected lung cancer-associated genes: 15

Gene-wise variant counts:
Hugo_Symbol
TP53     7
EGFR     3
BRAF     2
RB1      1
NF1      1
KEAP1    1
Name: count, dtype: int64

Prioritised candidate variants:


,Sample_ID,Hugo_Symbol,Chromosome,Start_Position,Reference_Allele,Tumor_Seq_Allele2,Variant_Classification,HGVSc,HGVSp_Short,gnomAD_AF,COSMIC,hotspot,t_depth,t_alt_count
847,TCGA-05-4382,BRAF,chr7,140753296,C,A,Missense_Mutation,c.1959G>T,p.L653F,NaN,NaN,N,170,13
6746,TCGA-55-8506,BRAF,chr7,140850111,C,A,Missense_Mutation,c.240G>T,p.E80D,NaN,NaN,N,131,15
791,TCGA-05-4382,EGFR,chr7,55152582,G,T,Missense_Mutation,c.665G>T,p.R222L,NaN,NaN,N,520,90
792,TCGA-05-4382,EGFR,chr7,55163734,G,C,Missense_Mutation,c.1633G>C,p.E545Q,NaN,NaN,N,223,17
6687,TCGA-55-8506,EGFR,chr7,55191822,T,G,Missense_Mutation,c.2573T>G,p.L858R,NaN,COSM6224,Y,144,68
7567,TCGA-55-8506,KEAP1,chr19,10486679,C,A,Missense_Mutation,c.1848G>T,p.Q616H,NaN,NaN,N,267,77
5646,TCGA-55-7994,NF1,chr17,31229130,A,T,Missense_Mutation,c.2515A>T,p.N839Y,NaN,NaN,N,35,8
3577,TCGA-17-Z031,RB1,chr13,48367603,G,T,Missense_Mutation,c.1049G>T,p.S350I,0.000004,NaN,N,69,48
1506,TCGA-05-4382,TP53,chr17,7673805,A,C,Missense_Mutation,c.815T>G,p.V272G,NaN,COSM3773298;COSM3773299;COSM44870;COSM562337,Y,44,9
3839,TCGA-17-Z031,TP53,chr17,7673838,C,-,Splice_Site,c.783-1del,p.X261_splice,NaN,NaN,Y,49,34



Evidence field completeness:
HGVSc : 15 non-missing records out of 15
HGVSp_Short : 15 non-missing records out of 15
gnomAD_AF : 3 non-missing records out of 15
COSMIC : 6 non-missing records out of 15
hotspot : 15 non-missing records out of 15
t_depth : 15 non-missing records out of 15
t_alt_count : 15 non-missing records out of 15


In [13]:

import os
import pandas as pd

output_dir = "/content/Task_3_Variant_Annotation/data/processed"
os.makedirs(output_dir, exist_ok=True)

# Select useful fields for the annotation dataset
selected_columns = [
    "Sample_ID",
    "Hugo_Symbol",
    "NCBI_Build",
    "Chromosome",
    "Start_Position",
    "End_Position",
    "Reference_Allele",
    "Tumor_Seq_Allele2",
    "Variant_Classification",
    "Variant_Type",
    "HGVSc",
    "HGVSp",
    "HGVSp_Short",
    "Transcript_ID",
    "gnomAD_AF",
    "COSMIC",
    "hotspot",
    "t_depth",
    "t_ref_count",
    "t_alt_count"
]

selected_columns = [
    col for col in selected_columns
    if col in driver_candidates.columns
]

final_candidates = driver_candidates[selected_columns].copy()

# Calculate variant allele fraction (VAF)
final_candidates["VAF"] = (
    pd.to_numeric(final_candidates["t_alt_count"], errors="coerce") /
    pd.to_numeric(final_candidates["t_depth"], errors="coerce")
)

# Save the selected variant dataset
output_file = os.path.join(
    output_dir, "LUAD_prioritised_variants.csv"
)

final_candidates.to_csv(output_file, index=False)

print("Prioritised variant dataset saved successfully!")
print("File:", output_file)
print("Total observations:", len(final_candidates))
print("Unique genes:", final_candidates["Hugo_Symbol"].nunique())
print("Unique genomic alterations:",
      final_candidates.drop_duplicates(
          ["Chromosome", "Start_Position",
           "Reference_Allele", "Tumor_Seq_Allele2"]
      ).shape[0])

print("\nCalculated VAF values:")
display(
    final_candidates[
        ["Sample_ID", "Hugo_Symbol", "HGVSp_Short", "VAF"]
    ]
)


Prioritised variant dataset saved successfully!
File: /content/Task_3_Variant_Annotation/data/processed/LUAD_prioritised_variants.csv
Total observations: 15
Unique genes: 6
Unique genomic alterations: 14

Calculated VAF values:


,Sample_ID,Hugo_Symbol,HGVSp_Short,VAF
791,TCGA-05-4382,EGFR,p.R222L,0.173077
792,TCGA-05-4382,EGFR,p.E545Q,0.076233
847,TCGA-05-4382,BRAF,p.L653F,0.076471
1506,TCGA-05-4382,TP53,p.V272G,0.204545
3577,TCGA-17-Z031,RB1,p.S350I,0.695652
3839,TCGA-17-Z031,TP53,p.X261_splice,0.693878
3840,TCGA-17-Z031,TP53,p.Q104H,0.454874
5634,TCGA-55-7994,TP53,p.R273L,0.235294
5646,TCGA-55-7994,NF1,p.N839Y,0.228571
6687,TCGA-55-8506,EGFR,p.L858R,0.472222


In [14]:

import requests
import pandas as pd
import json
import os
import time

# Input dataset
input_file = "/content/Task_3_Variant_Annotation/data/processed/LUAD_prioritised_variants.csv"

# Output folder
output_dir = "/content/Task_3_Variant_Annotation/results/annotated"
os.makedirs(output_dir, exist_ok=True)

variants = pd.read_csv(input_file)

# Prepare variants in Ensembl VEP input format
vep_inputs = []
variant_labels = []

for _, row in variants.iterrows():
    chrom = str(row["Chromosome"]).replace("chr", "")
    pos = int(row["Start_Position"])
    ref = str(row["Reference_Allele"])
    alt = str(row["Tumor_Seq_Allele2"])

    # VEP uses "-" to represent a deleted allele
    vep_line = f"{chrom} {pos} . {ref} {alt} . . ."

    vep_inputs.append(vep_line)
    variant_labels.append(
        f'{row["Sample_ID"]}_{row["Hugo_Symbol"]}_{row["HGVSp_Short"]}'
    )

print("Variants prepared for VEP:", len(vep_inputs))
print("\nExample inputs:")
for item in vep_inputs[:5]:
    print(item)

# Submit variants to Ensembl VEP
url = "https://rest.ensembl.org/vep/human/region"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

params = {
    "hgvs": 1,
    "canonical": 1,
    "mane": 1,
    "variant_class": 1
}

payload = {"variants": vep_inputs}

response = requests.post(
    url,
    headers=headers,
    params=params,
    json=payload,
    timeout=180
)

print("\nVEP HTTP status:", response.status_code)
response.raise_for_status()

vep_results = response.json()

print("VEP returned records:", len(vep_results))

# Save complete raw VEP response
json_file = os.path.join(output_dir, "LUAD_VEP_raw_results.json")

with open(json_file, "w") as f:
    json.dump(vep_results, f, indent=2)

print("Raw VEP results saved:", json_file)

# Extract a concise transcript annotation summary
summary = []

for label, result in zip(variant_labels, vep_results):
    transcript_data = result.get("transcript_consequences", [])

    # Prefer canonical transcript, otherwise first available transcript
    canonical = [
        t for t in transcript_data
        if t.get("canonical") == 1
    ]

    chosen = canonical[0] if canonical else (
        transcript_data[0] if transcript_data else {}
    )

    summary.append({
        "Variant_Label": label,
        "VEP_Gene": chosen.get("gene_symbol"),
        "Transcript_ID": chosen.get("transcript_id"),
        "VEP_Consequence": result.get("most_severe_consequence"),
        "VEP_Impact": chosen.get("impact"),
        "VEP_HGVSc": chosen.get("hgvsc"),
        "VEP_HGVSp": chosen.get("hgvsp"),
        "Canonical_Transcript": chosen.get("canonical"),
        "Assembly": result.get("assembly_name"),
        "VEP_ID": result.get("id")
    })

vep_summary = pd.DataFrame(summary)

csv_file = os.path.join(output_dir, "LUAD_VEP_annotation_summary.csv")
vep_summary.to_csv(csv_file, index=False)

print("\nVEP annotation summary:")
display(vep_summary)

print("\nSummary saved:", csv_file)


Variants prepared for VEP: 15

Example inputs:
7 55152582 . G T . . .
7 55163734 . G C . . .
7 140753296 . C A . . .
17 7673805 . A C . . .
13 48367603 . G T . . .

VEP HTTP status: 200
VEP returned records: 15
Raw VEP results saved: /content/Task_3_Variant_Annotation/results/annotated/LUAD_VEP_raw_results.json

VEP annotation summary:


,Variant_Label,VEP_Gene,Transcript_ID,VEP_Consequence,VEP_Impact,VEP_HGVSc,VEP_HGVSp,Canonical_Transcript,Assembly,VEP_ID
0,TCGA-05-4382_EGFR_p.R222L,EGFR,ENST00000275493,missense_variant,MODERATE,ENST00000275493.7:c.665G>T,ENSP00000275493.2:p.Arg222Leu,1,GRCh38,.
1,TCGA-05-4382_EGFR_p.E545Q,EGFR,ENST00000275493,missense_variant,MODERATE,ENST00000275493.7:c.1633G>C,ENSP00000275493.2:p.Glu545Gln,1,GRCh38,.
2,TCGA-05-4382_BRAF_p.L653F,BRAF,ENST00000646891,missense_variant,MODERATE,ENST00000646891.2:c.1839G>T,ENSP00000493543.1:p.Leu613Phe,1,GRCh38,.
3,TCGA-05-4382_TP53_p.V272G,TP53,ENST00000269305,missense_variant,MODERATE,ENST00000269305.9:c.815T>G,ENSP00000269305.4:p.Val272Gly,1,GRCh38,.
4,TCGA-17-Z031_RB1_p.S350I,RB1,ENST00000267163,missense_variant,MODERATE,ENST00000267163.6:c.1049G>T,ENSP00000267163.4:p.Ser350Ile,1,GRCh38,.
5,TCGA-17-Z031_TP53_p.X261_splice,TP53,ENST00000269305,splice_acceptor_variant,HIGH,ENST00000269305.9:c.783-1del,None,1,GRCh38,.
6,TCGA-17-Z031_TP53_p.Q104H,TP53,ENST00000269305,missense_variant,MODERATE,ENST00000269305.9:c.312G>T,ENSP00000269305.4:p.Gln104His,1,GRCh38,.
7,TCGA-55-7994_TP53_p.R273L,TP53,ENST00000269305,missense_variant,MODERATE,ENST00000269305.9:c.818G>T,ENSP00000269305.4:p.Arg273Leu,1,GRCh38,.
8,TCGA-55-7994_NF1_p.N839Y,NF1,ENST00000358273,missense_variant,MODERATE,ENST00000358273.9:c.2515A>T,ENSP00000351015.4:p.Asn839Tyr,1,GRCh38,.
9,TCGA-55-8506_EGFR_p.L858R,EGFR,ENST00000275493,missense_variant,MODERATE,ENST00000275493.7:c.2573T>G,ENSP00000275493.2:p.Leu858Arg,1,GRCh38,.



Summary saved: /content/Task_3_Variant_Annotation/results/annotated/LUAD_VEP_annotation_summary.csv


In [15]:

import pandas as pd
import os

base_dir = "/content/Task_3_Variant_Annotation"

maf_file = os.path.join(
    base_dir, "data/processed/LUAD_prioritised_variants.csv"
)

vep_file = os.path.join(
    base_dir, "results/annotated/LUAD_VEP_annotation_summary.csv"
)

maf_df = pd.read_csv(maf_file)
vep_df = pd.read_csv(vep_file)

# Create the same label in both datasets
maf_df["Variant_Label"] = (
    maf_df["Sample_ID"].astype(str) + "_" +
    maf_df["Hugo_Symbol"].astype(str) + "_" +
    maf_df["HGVSp_Short"].astype(str)
)

# Combine original MAF and VEP annotations
merged_df = pd.merge(
    maf_df,
    vep_df,
    on="Variant_Label",
    how="left",
    validate="one_to_one"
)

print("Original MAF observations:", len(maf_df))
print("VEP annotations:", len(vep_df))
print("Merged observations:", len(merged_df))
print("Missing VEP consequences:",
      merged_df["VEP_Consequence"].isna().sum())

# Save the integrated annotation table
output_file = os.path.join(
    base_dir, "results/annotated/LUAD_integrated_annotation.csv"
)

merged_df.to_csv(output_file, index=False)

print("\nIntegrated annotation saved:")
print(output_file)

display(
    merged_df[
        [
            "Sample_ID", "Hugo_Symbol", "HGVSp_Short",
            "VEP_Consequence", "VEP_Impact",
            "HGVSc", "VEP_HGVSc", "HGVSp", "VEP_HGVSp"
        ]
    ]
)


Original MAF observations: 15
VEP annotations: 15
Merged observations: 15
Missing VEP consequences: 0

Integrated annotation saved:
/content/Task_3_Variant_Annotation/results/annotated/LUAD_integrated_annotation.csv


,Sample_ID,Hugo_Symbol,HGVSp_Short,VEP_Consequence,VEP_Impact,HGVSc,VEP_HGVSc,HGVSp,VEP_HGVSp
0,TCGA-05-4382,EGFR,p.R222L,missense_variant,MODERATE,c.665G>T,ENST00000275493.7:c.665G>T,p.Arg222Leu,ENSP00000275493.2:p.Arg222Leu
1,TCGA-05-4382,EGFR,p.E545Q,missense_variant,MODERATE,c.1633G>C,ENST00000275493.7:c.1633G>C,p.Glu545Gln,ENSP00000275493.2:p.Glu545Gln
2,TCGA-05-4382,BRAF,p.L653F,missense_variant,MODERATE,c.1959G>T,ENST00000646891.2:c.1839G>T,p.Leu653Phe,ENSP00000493543.1:p.Leu613Phe
3,TCGA-05-4382,TP53,p.V272G,missense_variant,MODERATE,c.815T>G,ENST00000269305.9:c.815T>G,p.Val272Gly,ENSP00000269305.4:p.Val272Gly
4,TCGA-17-Z031,RB1,p.S350I,missense_variant,MODERATE,c.1049G>T,ENST00000267163.6:c.1049G>T,p.Ser350Ile,ENSP00000267163.4:p.Ser350Ile
5,TCGA-17-Z031,TP53,p.X261_splice,splice_acceptor_variant,HIGH,c.783-1del,ENST00000269305.9:c.783-1del,NaN,NaN
6,TCGA-17-Z031,TP53,p.Q104H,missense_variant,MODERATE,c.312G>T,ENST00000269305.9:c.312G>T,p.Gln104His,ENSP00000269305.4:p.Gln104His
7,TCGA-55-7994,TP53,p.R273L,missense_variant,MODERATE,c.818G>T,ENST00000269305.9:c.818G>T,p.Arg273Leu,ENSP00000269305.4:p.Arg273Leu
8,TCGA-55-7994,NF1,p.N839Y,missense_variant,MODERATE,c.2515A>T,ENST00000358273.9:c.2515A>T,p.Asn839Tyr,ENSP00000351015.4:p.Asn839Tyr
9,TCGA-55-8506,EGFR,p.L858R,missense_variant,MODERATE,c.2573T>G,ENST00000275493.7:c.2573T>G,p.Leu858Arg,ENSP00000275493.2:p.Leu858Arg


In [17]:
print(df.columns.tolist())

['Sample_ID', 'Hugo_Symbol', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Reference_Allele', 'Tumor_Seq_Allele2', 'Variant_Classification', 'Variant_Type', 'HGVSc', 'HGVSp', 'HGVSp_Short', 'Transcript_ID_x', 'gnomAD_AF', 'COSMIC', 'hotspot', 't_depth', 't_ref_count', 't_alt_count', 'VAF', 'Variant_Label', 'VEP_Gene', 'Transcript_ID_y', 'VEP_Consequence', 'VEP_Impact', 'VEP_HGVSc', 'VEP_HGVSp', 'Canonical_Transcript', 'Assembly', 'VEP_ID']


In [18]:
braf = df[df["Hugo_Symbol"] == "BRAF"]

display(
    braf[
        [
            "Sample_ID",
            "Chromosome",
            "Start_Position",
            "End_Position",
            "Reference_Allele",
            "Tumor_Seq_Allele2",
            "Transcript_ID_x",
            "HGVSc",
            "HGVSp_Short",
            "Transcript_ID_y",
            "VEP_HGVSc",
            "VEP_HGVSp"
        ]
    ]
)

,Sample_ID,Chromosome,Start_Position,End_Position,Reference_Allele,Tumor_Seq_Allele2,Transcript_ID_x,HGVSc,HGVSp_Short,Transcript_ID_y,VEP_HGVSc,VEP_HGVSp
2,TCGA-05-4382,chr7,140753296,140753296,C,A,ENST00000288602,c.1959G>T,p.L653F,ENST00000646891,ENST00000646891.2:c.1839G>T,ENSP00000493543.1:p.Leu613Phe
10,TCGA-55-8506,chr7,140850111,140850111,C,A,ENST00000288602,c.240G>T,p.E80D,ENST00000646891,ENST00000646891.2:c.240G>T,ENSP00000493543.1:p.Glu80Asp


In [19]:
import requests
import pandas as pd

variants = [
    {
        "sample": "TCGA-05-4382",
        "region": "7 140753296 . C A . . ."
    },
    {
        "sample": "TCGA-55-8506",
        "region": "7 140850111 . C A . . ."
    }
]

url = "https://rest.ensembl.org/vep/human/region"

params = {
    "hgvs": 1,
    "transcript_id": "ENST00000288602",
    "transcript_version": 1
}

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

response = requests.post(
    url,
    params=params,
    json={"variants": [v["region"] for v in variants]},
    headers=headers,
    timeout=60
)

print("API status:", response.status_code)

if response.status_code == 200:
    results = response.json()

    for sample, result in zip(variants, results):
        print("\nSample:", sample["sample"])
        print("Variant:", result.get("input"))

        for tc in result.get("transcript_consequences", []):
            print({
                "transcript": tc.get("transcript_id"),
                "consequence": tc.get("consequence_terms"),
                "HGVSc": tc.get("hgvsc"),
                "HGVSp": tc.get("hgvsp"),
                "canonical": tc.get("canonical")
            })
else:
    print(response.text[:2000])

API status: 200

Sample: TCGA-05-4382
Variant: 7 140753296 . C A . . .
{'transcript': 'ENST00000288602.12', 'consequence': ['missense_variant'], 'HGVSc': 'ENST00000288602.12:c.1959G>T', 'HGVSp': 'ENSP00000288602.7:p.Leu653Phe', 'canonical': None}

Sample: TCGA-55-8506
Variant: 7 140850111 . C A . . .
{'transcript': 'ENST00000288602.12', 'consequence': ['missense_variant', 'splice_region_variant'], 'HGVSc': 'ENST00000288602.12:c.240G>T', 'HGVSp': 'ENSP00000288602.7:p.Glu80Asp', 'canonical': None}


In [20]:
import os
import pandas as pd

base_dir = "/content/Task_3_Variant_Annotation"

verification = pd.DataFrame([
    {
        "Sample_ID": "TCGA-05-4382",
        "Gene": "BRAF",
        "MAF_Transcript": "ENST00000288602",
        "Verified_Transcript": "ENST00000288602.12",
        "MAF_HGVSc": "c.1959G>T",
        "Verified_HGVSc": "c.1959G>T",
        "MAF_HGVSp": "p.Leu653Phe",
        "Verified_HGVSp": "p.Leu653Phe",
        "Verification": "Match"
    },
    {
        "Sample_ID": "TCGA-55-8506",
        "Gene": "BRAF",
        "MAF_Transcript": "ENST00000288602",
        "Verified_Transcript": "ENST00000288602.12",
        "MAF_HGVSc": "c.240G>T",
        "Verified_HGVSc": "c.240G>T",
        "MAF_HGVSp": "p.Glu80Asp",
        "Verified_HGVSp": "p.Glu80Asp",
        "Verification": "Match"
    }
])

output_file = os.path.join(
    base_dir,
    "results/annotated/BRAF_transcript_verification.csv"
)

verification.to_csv(output_file, index=False)

display(verification)

print("Verification saved:", output_file)

,Sample_ID,Gene,MAF_Transcript,Verified_Transcript,MAF_HGVSc,Verified_HGVSc,MAF_HGVSp,Verified_HGVSp,Verification
0,TCGA-05-4382,BRAF,ENST00000288602,ENST00000288602.12,c.1959G>T,c.1959G>T,p.Leu653Phe,p.Leu653Phe,Match
1,TCGA-55-8506,BRAF,ENST00000288602,ENST00000288602.12,c.240G>T,c.240G>T,p.Glu80Asp,p.Glu80Asp,Match


Verification saved: /content/Task_3_Variant_Annotation/results/annotated/BRAF_transcript_verification.csv


In [21]:
import pandas as pd
import os

base_dir = "/content/Task_3_Variant_Annotation"

df = pd.read_csv(
    os.path.join(
        base_dir,
        "results/annotated/LUAD_integrated_annotation.csv"
    )
)

clinical_df = df[
    [
        "Sample_ID",
        "Hugo_Symbol",
        "Chromosome",
        "Start_Position",
        "Reference_Allele",
        "Tumor_Seq_Allele2",
        "HGVSc",
        "HGVSp_Short",
        "VEP_Consequence",
        "VEP_Impact",
        "gnomAD_AF",
        "COSMIC",
        "hotspot",
        "VAF"
    ]
].copy()

clinical_df["ClinVar_Evidence"] = "To be assessed"
clinical_df["Oncology_Knowledgebase_Evidence"] = "To be assessed"
clinical_df["AMP_ASCO_CAP_Tier"] = "Pending evidence review"
clinical_df["Therapeutic_Relevance"] = "To be assessed"
clinical_df["Evidence_Notes"] = ""

output_file = os.path.join(
    base_dir,
    "results/tables/LUAD_clinical_evidence_assessment.csv"
)

clinical_df.to_csv(output_file, index=False)

print("Clinical evidence table created.")
print("Total variant observations:", len(clinical_df))
print("Saved:", output_file)

display(clinical_df.head())

Clinical evidence table created.
Total variant observations: 15
Saved: /content/Task_3_Variant_Annotation/results/tables/LUAD_clinical_evidence_assessment.csv


,Sample_ID,Hugo_Symbol,Chromosome,Start_Position,Reference_Allele,Tumor_Seq_Allele2,HGVSc,HGVSp_Short,VEP_Consequence,VEP_Impact,gnomAD_AF,COSMIC,hotspot,VAF,ClinVar_Evidence,Oncology_Knowledgebase_Evidence,AMP_ASCO_CAP_Tier,Therapeutic_Relevance,Evidence_Notes
0,TCGA-05-4382,EGFR,chr7,55152582,G,T,c.665G>T,p.R222L,missense_variant,MODERATE,NaN,NaN,N,0.173077,To be assessed,To be assessed,Pending evidence review,To be assessed,
1,TCGA-05-4382,EGFR,chr7,55163734,G,C,c.1633G>C,p.E545Q,missense_variant,MODERATE,NaN,NaN,N,0.076233,To be assessed,To be assessed,Pending evidence review,To be assessed,
2,TCGA-05-4382,BRAF,chr7,140753296,C,A,c.1959G>T,p.L653F,missense_variant,MODERATE,NaN,NaN,N,0.076471,To be assessed,To be assessed,Pending evidence review,To be assessed,
3,TCGA-05-4382,TP53,chr17,7673805,A,C,c.815T>G,p.V272G,missense_variant,MODERATE,NaN,COSM3773298;COSM3773299;COSM44870;COSM562337,Y,0.204545,To be assessed,To be assessed,Pending evidence review,To be assessed,
4,TCGA-17-Z031,RB1,chr13,48367603,G,T,c.1049G>T,p.S350I,missense_variant,MODERATE,0.000004,NaN,N,0.695652,To be assessed,To be assessed,Pending evidence review,To be assessed,


In [22]:
import pandas as pd
import os

base_dir = "/content/Task_3_Variant_Annotation"

file_path = os.path.join(
    base_dir,
    "results/tables/LUAD_clinical_evidence_assessment.csv"
)

clinical_df = pd.read_csv(file_path)

# EGFR p.L858R
mask_egfr = (
    (clinical_df["Hugo_Symbol"] == "EGFR") &
    (clinical_df["HGVSp_Short"] == "p.L858R")
)

clinical_df.loc[mask_egfr, "ClinVar_Evidence"] = (
    "Somatic clinical impact: Tier I (Strong) for NSCLC; "
    "associated with lung adenocarcinoma and TKI response"
)

clinical_df.loc[mask_egfr, "Oncology_Knowledgebase_Evidence"] = (
    "ClinVar expert-panel reviewed somatic evidence; "
    "multiple submitters support Tier I for NSCLC"
)

clinical_df.loc[mask_egfr, "AMP_ASCO_CAP_Tier"] = "Tier I - Strong"

clinical_df.loc[mask_egfr, "Therapeutic_Relevance"] = (
    "Established EGFR-targeted therapy biomarker in NSCLC; "
    "clinical interpretation depends on tumor context"
)

clinical_df.loc[mask_egfr, "Evidence_Notes"] = (
    "ClinVar Variation ID 16609. Somatic Tier I evidence is "
    "for NSCLC; not an individual treatment recommendation."
)

# TP53 p.V272G
mask_tp53 = (
    (clinical_df["Hugo_Symbol"] == "TP53") &
    (clinical_df["HGVSp_Short"] == "p.V272G")
)

clinical_df.loc[mask_tp53, "ClinVar_Evidence"] = (
    "Conflicting germline classifications; no submitted "
    "somatic clinical impact or oncogenicity data"
)

clinical_df.loc[mask_tp53, "Oncology_Knowledgebase_Evidence"] = (
    "No confirmed LUAD-specific clinical evidence established"
)

clinical_df.loc[mask_tp53, "AMP_ASCO_CAP_Tier"] = (
    "Tier III - provisional; somatic significance uncertain"
)

clinical_df.loc[mask_tp53, "Therapeutic_Relevance"] = (
    "No established variant-specific therapeutic relevance "
    "identified from the reviewed evidence"
)

clinical_df.loc[mask_tp53, "Evidence_Notes"] = (
    "Do not transfer germline pathogenicity classifications "
    "to somatic AMP/ASCO/CAP interpretation."
)

clinical_df.to_csv(file_path, index=False)

print("Clinical evidence updated.")
display(
    clinical_df[
        clinical_df["HGVSp_Short"].isin(["p.L858R", "p.V272G"])
    ][
        [
            "Sample_ID",
            "Hugo_Symbol",
            "HGVSp_Short",
            "ClinVar_Evidence",
            "AMP_ASCO_CAP_Tier",
            "Therapeutic_Relevance"
        ]
    ]
)

Clinical evidence updated.


/tmp/ipykernel_2819/3774809598.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ClinVar Variation ID 16609. Somatic Tier I evidence is for NSCLC; not an individual treatment recommendation.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  clinical_df.loc[mask_egfr, "Evidence_Notes"] = (


,Sample_ID,Hugo_Symbol,HGVSp_Short,ClinVar_Evidence,AMP_ASCO_CAP_Tier,Therapeutic_Relevance
3,TCGA-05-4382,TP53,p.V272G,Conflicting germline classifications; no submi...,Tier III - provisional; somatic significance u...,No established variant-specific therapeutic re...
9,TCGA-55-8506,EGFR,p.L858R,Somatic clinical impact: Tier I (Strong) for N...,Tier I - Strong,Established EGFR-targeted therapy biomarker in...


In [23]:
# Convert text-based evidence columns to string type
text_columns = [
    "ClinVar_Evidence",
    "Oncology_Knowledgebase_Evidence",
    "AMP_ASCO_CAP_Tier",
    "Therapeutic_Relevance",
    "Evidence_Notes"
]

for col in text_columns:
    clinical_df[col] = clinical_df[col].fillna("").astype(str)

clinical_df.to_csv(file_path, index=False)

print("Data types corrected and clinical evidence table saved.")
print(clinical_df[text_columns].dtypes)

Data types corrected and clinical evidence table saved.
ClinVar_Evidence                   object
Oncology_Knowledgebase_Evidence    object
AMP_ASCO_CAP_Tier                  object
Therapeutic_Relevance              object
Evidence_Notes                     object
dtype: object


In [24]:
mask_tp53_r273l = (
    (clinical_df["Hugo_Symbol"] == "TP53") &
    (clinical_df["HGVSp_Short"] == "p.R273L")
)

clinical_df.loc[mask_tp53_r273l, "ClinVar_Evidence"] = (
    "ClinVar Variation ID 376655: oncogenic; somatic Tier I "
    "reported for embryonal rhabdomyosarcoma only. "
    "LUAD-specific record has no somatic clinical-impact tier."
)

clinical_df.loc[mask_tp53_r273l, "Oncology_Knowledgebase_Evidence"] = (
    "OncoKB: likely oncogenic, loss-of-function; "
    "no established LUAD-specific therapeutic evidence."
)

clinical_df.loc[mask_tp53_r273l, "AMP_ASCO_CAP_Tier"] = (
    "Tier III - unknown clinical significance in LUAD"
)

clinical_df.loc[mask_tp53_r273l, "Therapeutic_Relevance"] = (
    "No established LUAD-specific targeted therapy identified."
)

clinical_df.loc[mask_tp53_r273l, "Evidence_Notes"] = (
    "TP53 p.R273L is a known oncogenic alteration. "
    "Do not transfer the ClinVar Tier I classification "
    "from embryonal rhabdomyosarcoma to LUAD. "
    "The LUAD-specific clinical impact remains unestablished."
)

clinical_df.to_csv(file_path, index=False)

print("TP53 p.R273L evidence updated.")
print("Number of observations updated:", mask_tp53_r273l.sum())

display(
    clinical_df.loc[
        mask_tp53_r273l,
        [
            "Sample_ID",
            "Hugo_Symbol",
            "HGVSp_Short",
            "AMP_ASCO_CAP_Tier",
            "Therapeutic_Relevance"
        ]
    ]
)

TP53 p.R273L evidence updated.
Number of observations updated: 2


,Sample_ID,Hugo_Symbol,HGVSp_Short,AMP_ASCO_CAP_Tier,Therapeutic_Relevance
7,TCGA-55-7994,TP53,p.R273L,Tier III - unknown clinical significance in LUAD,No established LUAD-specific targeted therapy ...
13,TCGA-L9-A7SV,TP53,p.R273L,Tier III - unknown clinical significance in LUAD,No established LUAD-specific targeted therapy ...


In [25]:
# TP53 p.G154V
mask_g154v = (
    (clinical_df["Hugo_Symbol"] == "TP53") &
    (clinical_df["HGVSp_Short"] == "p.G154V")
)

clinical_df.loc[mask_g154v, "ClinVar_Evidence"] = (
    "ClinVar Variation ID 651769: conflicting germline "
    "classifications; no submitted somatic clinical impact "
    "or oncogenicity classification."
)

clinical_df.loc[mask_g154v, "Oncology_Knowledgebase_Evidence"] = (
    "Functional evidence of impaired TP53 activity reported; "
    "LUAD-specific clinical evidence not established."
)

clinical_df.loc[mask_g154v, "AMP_ASCO_CAP_Tier"] = (
    "Tier III - provisional; LUAD clinical significance uncertain"
)

clinical_df.loc[mask_g154v, "Therapeutic_Relevance"] = (
    "No established LUAD-specific targeted therapy identified."
)

clinical_df.loc[mask_g154v, "Evidence_Notes"] = (
    "ClinVar germline evidence must not be interpreted as a "
    "somatic clinical-impact classification."
)

# TP53 p.I195F
mask_i195f = (
    (clinical_df["Hugo_Symbol"] == "TP53") &
    (clinical_df["HGVSp_Short"] == "p.I195F")
)

clinical_df.loc[mask_i195f, "ClinVar_Evidence"] = (
    "ClinVar Variation ID 376617: functional evidence reported; "
    "somatic pathogenic assertion exists for esophageal carcinoma, "
    "not LUAD."
)

clinical_df.loc[mask_i195f, "Oncology_Knowledgebase_Evidence"] = (
    "Reported TP53 functional impairment; no established "
    "LUAD-specific therapeutic evidence."
)

clinical_df.loc[mask_i195f, "AMP_ASCO_CAP_Tier"] = (
    "Tier III - provisional; LUAD clinical significance uncertain"
)

clinical_df.loc[mask_i195f, "Therapeutic_Relevance"] = (
    "No established LUAD-specific targeted therapy identified."
)

clinical_df.loc[mask_i195f, "Evidence_Notes"] = (
    "Evidence from esophageal carcinoma and hereditary cancer "
    "does not establish clinical actionability in LUAD."
)

clinical_df.to_csv(file_path, index=False)

print("TP53 evidence updates completed.")

display(
    clinical_df.loc[
        mask_g154v | mask_i195f,
        [
            "Sample_ID",
            "HGVSp_Short",
            "ClinVar_Evidence",
            "AMP_ASCO_CAP_Tier"
        ]
    ]
)

TP53 evidence updates completed.


,Sample_ID,HGVSp_Short,ClinVar_Evidence,AMP_ASCO_CAP_Tier
11,TCGA-55-8506,p.G154V,ClinVar Variation ID 651769: conflicting germl...,Tier III - provisional; LUAD clinical signific...
14,TCGA-L9-A7SV,p.I195F,ClinVar Variation ID 376617: functional eviden...,Tier III - provisional; LUAD clinical signific...


In [26]:
remaining_genes = ["EGFR", "BRAF", "KEAP1", "RB1", "NF1"]

remaining_df = clinical_df[
    clinical_df["Hugo_Symbol"].isin(remaining_genes)
].copy()

display(
    remaining_df[
        [
            "Sample_ID",
            "Hugo_Symbol",
            "Chromosome",
            "Start_Position",
            "Reference_Allele",
            "Tumor_Seq_Allele2",
            "HGVSc",
            "HGVSp_Short",
            "COSMIC",
            "hotspot",
            "gnomAD_AF",
            "VAF"
        ]
    ].sort_values(["Hugo_Symbol", "HGVSp_Short"])
)

print("Remaining gene observations:", len(remaining_df))

,Sample_ID,Hugo_Symbol,Chromosome,Start_Position,Reference_Allele,Tumor_Seq_Allele2,HGVSc,HGVSp_Short,COSMIC,hotspot,gnomAD_AF,VAF
10,TCGA-55-8506,BRAF,chr7,140850111,C,A,c.240G>T,p.E80D,NaN,N,NaN,0.114504
2,TCGA-05-4382,BRAF,chr7,140753296,C,A,c.1959G>T,p.L653F,NaN,N,NaN,0.076471
1,TCGA-05-4382,EGFR,chr7,55163734,G,C,c.1633G>C,p.E545Q,NaN,N,NaN,0.076233
9,TCGA-55-8506,EGFR,chr7,55191822,T,G,c.2573T>G,p.L858R,COSM6224,Y,NaN,0.472222
0,TCGA-05-4382,EGFR,chr7,55152582,G,T,c.665G>T,p.R222L,NaN,N,NaN,0.173077
12,TCGA-55-8506,KEAP1,chr19,10486679,C,A,c.1848G>T,p.Q616H,NaN,N,NaN,0.288390
8,TCGA-55-7994,NF1,chr17,31229130,A,T,c.2515A>T,p.N839Y,NaN,N,NaN,0.228571
4,TCGA-17-Z031,RB1,chr13,48367603,G,T,c.1049G>T,p.S350I,NaN,N,0.000004,0.695652


Remaining gene observations: 8


In [27]:
# Preliminary evidence assessment for variants without
# confirmed LUAD-specific clinical actionability.

preliminary_updates = {
    ("EGFR", "p.R222L"): (
        "No exact matching somatic ClinVar classification verified.",
        "No established LUAD-specific evidence verified.",
        "Tier III - provisional",
        "No established variant-specific therapeutic relevance identified.",
        "Do not confuse with EGFR c.665G>A (p.R222H) or other substitutions at codon 222."
    ),

    ("EGFR", "p.E545Q"): (
        "No exact matching somatic ClinVar classification verified.",
        "No established LUAD-specific evidence verified.",
        "Tier III - provisional",
        "No established variant-specific therapeutic relevance identified.",
        "Exact nucleotide and amino-acid substitution require variant-specific evidence."
    ),

    ("BRAF", "p.L653F"): (
        "No exact matching somatic ClinVar classification verified.",
        "No established LUAD-specific evidence verified.",
        "Tier III - provisional",
        "No established variant-specific therapeutic relevance identified.",
        "Transcript-dependent HGVS verified; retain original and canonical transcript annotations."
    ),

    ("BRAF", "p.E80D"): (
        "No exact matching somatic ClinVar classification verified.",
        "No established LUAD-specific evidence verified.",
        "Tier III - provisional",
        "No established variant-specific therapeutic relevance identified.",
        "Do not infer therapeutic relevance from other BRAF mutations such as V600E."
    ),

    ("KEAP1", "p.Q616H"): (
        "Exact variant record requires transcript-level verification before interpretation.",
        "No established LUAD-specific therapeutic evidence verified.",
        "Tier III - provisional",
        "Potential relevance to KEAP1 pathway biology; variant-specific actionability unestablished.",
        "KEAP1 alterations may be relevant to lung cancer biology, but gene-level evidence does not establish this variant's clinical tier."
    ),

    ("NF1", "p.N839Y"): (
        "No exact matching somatic ClinVar classification verified.",
        "No established LUAD-specific evidence verified.",
        "Tier III - provisional",
        "No established variant-specific therapeutic relevance identified.",
        "Variant-specific functional and clinical evidence remains to be assessed."
    ),

    ("RB1", "p.S350I"): (
        "No exact matching somatic ClinVar classification verified.",
        "No established LUAD-specific evidence verified.",
        "Tier III - provisional",
        "No established variant-specific therapeutic relevance identified.",
        "Do not confuse with other RB1 substitutions at codon 350."
    )
}

for (gene, protein), values in preliminary_updates.items():
    mask = (
        (clinical_df["Hugo_Symbol"] == gene) &
        (clinical_df["HGVSp_Short"] == protein)
    )

    clinical_df.loc[mask, "ClinVar_Evidence"] = values[0]
    clinical_df.loc[mask, "Oncology_Knowledgebase_Evidence"] = values[1]
    clinical_df.loc[mask, "AMP_ASCO_CAP_Tier"] = values[2]
    clinical_df.loc[mask, "Therapeutic_Relevance"] = values[3]
    clinical_df.loc[mask, "Evidence_Notes"] = values[4]

clinical_df.to_csv(file_path, index=False)

print("Preliminary evidence assessment saved.")

display(
    clinical_df[
        clinical_df["Hugo_Symbol"].isin(
            ["EGFR", "BRAF", "KEAP1", "NF1", "RB1"]
        )
    ][
        [
            "Hugo_Symbol",
            "HGVSp_Short",
            "AMP_ASCO_CAP_Tier",
            "Therapeutic_Relevance"
        ]
    ]
)

Preliminary evidence assessment saved.


,Hugo_Symbol,HGVSp_Short,AMP_ASCO_CAP_Tier,Therapeutic_Relevance
0,EGFR,p.R222L,Tier III - provisional,No established variant-specific therapeutic re...
1,EGFR,p.E545Q,Tier III - provisional,No established variant-specific therapeutic re...
2,BRAF,p.L653F,Tier III - provisional,No established variant-specific therapeutic re...
4,RB1,p.S350I,Tier III - provisional,No established variant-specific therapeutic re...
8,NF1,p.N839Y,Tier III - provisional,No established variant-specific therapeutic re...
9,EGFR,p.L858R,Tier I - Strong,Established EGFR-targeted therapy biomarker in...
10,BRAF,p.E80D,Tier III - provisional,No established variant-specific therapeutic re...
12,KEAP1,p.Q616H,Tier III - provisional,Potential relevance to KEAP1 pathway biology; ...


In [28]:
mask_splice = (
    (clinical_df["Hugo_Symbol"] == "TP53") &
    (clinical_df["HGVSp_Short"] == "p.X261_splice")
)

clinical_df.loc[mask_splice, "ClinVar_Evidence"] = (
    "No exact matching ClinVar classification verified. "
    "Nearby TP53 splice-site substitutions are different variants."
)

clinical_df.loc[mask_splice, "Oncology_Knowledgebase_Evidence"] = (
    "Exact-variant somatic clinical evidence not established."
)

clinical_df.loc[mask_splice, "AMP_ASCO_CAP_Tier"] = (
    "Tier III - provisional; LUAD clinical significance uncertain"
)

clinical_df.loc[mask_splice, "Therapeutic_Relevance"] = (
    "Potential loss of normal TP53 function through splice disruption; "
    "no established LUAD-specific targeted therapy."
)

clinical_df.loc[mask_splice, "Evidence_Notes"] = (
    "VEP predicts HIGH-impact splice_acceptor_variant. "
    "Do not transfer ClinVar classifications from different "
    "TP53 splice-site substitutions."
)

clinical_df.to_csv(file_path, index=False)

print("TP53 splice-site evidence updated.")
print("Observations updated:", mask_splice.sum())

display(
    clinical_df.loc[
        mask_splice,
        [
            "Sample_ID",
            "HGVSc",
            "VEP_Consequence",
            "VEP_Impact",
            "AMP_ASCO_CAP_Tier",
            "Evidence_Notes"
        ]
    ]
)

TP53 splice-site evidence updated.
Observations updated: 1


,Sample_ID,HGVSc,VEP_Consequence,VEP_Impact,AMP_ASCO_CAP_Tier,Evidence_Notes
5,TCGA-17-Z031,c.783-1del,splice_acceptor_variant,HIGH,Tier III - provisional; LUAD clinical signific...,VEP predicts HIGH-impact splice_acceptor_varia...


In [29]:
import pandas as pd
import os

base_dir = "/content/Task_3_Variant_Annotation"

clinical_df = pd.read_csv(
    os.path.join(
        base_dir,
        "results/tables/LUAD_clinical_evidence_assessment.csv"
    )
)

print("========== DATASET QUALITY CHECK ==========")

print("\nTotal variant observations:", len(clinical_df))

print("\nMissing essential annotations:")
essential_columns = [
    "Sample_ID",
    "Hugo_Symbol",
    "Chromosome",
    "Start_Position",
    "HGVSc",
    "HGVSp_Short",
    "VEP_Consequence",
    "VEP_Impact"
]

print(clinical_df[essential_columns].isna().sum())

print("\nMissing clinical evidence fields:")
evidence_columns = [
    "ClinVar_Evidence",
    "Oncology_Knowledgebase_Evidence",
    "AMP_ASCO_CAP_Tier",
    "Therapeutic_Relevance",
    "Evidence_Notes"
]

print(clinical_df[evidence_columns].isna().sum())

print("\nAMP/ASCO/CAP tier distribution:")
print(clinical_df["AMP_ASCO_CAP_Tier"].value_counts(dropna=False))

print("\nVariants by gene:")
print(clinical_df["Hugo_Symbol"].value_counts())

print("\nVEP impact distribution:")
print(clinical_df["VEP_Impact"].value_counts(dropna=False))

print("\nDuplicate genomic observations:")
print(
    clinical_df.duplicated(
        subset=[
            "Sample_ID",
            "Chromosome",
            "Start_Position",
            "Reference_Allele",
            "Tumor_Seq_Allele2"
        ]
    ).sum()
)

print("\n========== QUALITY CHECK COMPLETE ==========")

========== DATASET QUALITY CHECK ==========

Total variant observations: 15

Missing essential annotations:
Sample_ID          0
Hugo_Symbol        0
Chromosome         0
Start_Position     0
HGVSc              0
HGVSp_Short        0
VEP_Consequence    0
VEP_Impact         0
dtype: int64

Missing clinical evidence fields:
ClinVar_Evidence                   0
Oncology_Knowledgebase_Evidence    0
AMP_ASCO_CAP_Tier                  0
Therapeutic_Relevance              0
Evidence_Notes                     1
dtype: int64

AMP/ASCO/CAP tier distribution:
AMP_ASCO_CAP_Tier
Tier III - provisional                                          7
Tier III - provisional; LUAD clinical significance uncertain    3
Tier III - unknown clinical significance in LUAD                2
Tier III - provisional; somatic significance uncertain          1
Pending evidence review                                         1
Tier I - Strong                                                 1
Name: count, dtype: int64

Vari

In [30]:
# Identify records requiring final clinical-evidence review

review_mask = (
    clinical_df["AMP_ASCO_CAP_Tier"]
    .astype(str)
    .str.contains("Pending", case=False, na=False)
    |
    clinical_df["Evidence_Notes"].isna()
    |
    clinical_df["Evidence_Notes"].astype(str).str.strip().eq("")
)

review_columns = [
    "Sample_ID",
    "Hugo_Symbol",
    "Chromosome",
    "Start_Position",
    "HGVSc",
    "HGVSp_Short",
    "VEP_Consequence",
    "ClinVar_Evidence",
    "Oncology_Knowledgebase_Evidence",
    "AMP_ASCO_CAP_Tier",
    "Evidence_Notes"
]

print("Records requiring attention:", review_mask.sum())

display(
    clinical_df.loc[review_mask, review_columns]
)

Records requiring attention: 1


,Sample_ID,Hugo_Symbol,Chromosome,Start_Position,HGVSc,HGVSp_Short,VEP_Consequence,ClinVar_Evidence,Oncology_Knowledgebase_Evidence,AMP_ASCO_CAP_Tier,Evidence_Notes
6,TCGA-17-Z031,TP53,chr17,7676057,c.312G>T,p.Q104H,missense_variant,To be assessed,To be assessed,Pending evidence review,NaN


In [31]:
mask_q104h = (
    (clinical_df["Hugo_Symbol"] == "TP53") &
    (clinical_df["HGVSc"] == "c.312G>T") &
    (clinical_df["Sample_ID"] == "TCGA-17-Z031")
)

clinical_df.loc[mask_q104h, "ClinVar_Evidence"] = (
    "Exact variant identity confirmed by ClinGen Allele Registry. "
    "No exact cancer-specific clinical significance assertion verified "
    "in the evidence reviewed."
)

clinical_df.loc[mask_q104h, "Oncology_Knowledgebase_Evidence"] = (
    "No established LUAD-specific therapeutic evidence verified "
    "for this exact substitution."
)

clinical_df.loc[mask_q104h, "AMP_ASCO_CAP_Tier"] = (
    "Tier III - provisional; LUAD clinical significance uncertain"
)

clinical_df.loc[mask_q104h, "Therapeutic_Relevance"] = (
    "TP53 missense alteration with uncertain clinical significance "
    "in LUAD; no established variant-specific targeted therapy."
)

clinical_df.loc[mask_q104h, "Evidence_Notes"] = (
    "ClinGen Allele Registry confirms NM_000546.6:c.312G>T, "
    "p.Gln104His at GRCh38 chr17:7676057 C>A. "
    "No exact LUAD-specific clinical assertion verified. "
    "Do not confuse with the different synonymous c.312G>A variant."
)

clinical_df.to_csv(file_path, index=False)

print("TP53 p.Q104H evidence updated.")
print("Observations updated:", mask_q104h.sum())

display(
    clinical_df.loc[mask_q104h, review_columns]
)

TP53 p.Q104H evidence updated.
Observations updated: 1


,Sample_ID,Hugo_Symbol,Chromosome,Start_Position,HGVSc,HGVSp_Short,VEP_Consequence,ClinVar_Evidence,Oncology_Knowledgebase_Evidence,AMP_ASCO_CAP_Tier,Evidence_Notes
6,TCGA-17-Z031,TP53,chr17,7676057,c.312G>T,p.Q104H,missense_variant,Exact variant identity confirmed by ClinGen Al...,No established LUAD-specific therapeutic evide...,Tier III - provisional; LUAD clinical signific...,ClinGen Allele Registry confirms NM_000546.6:c...


In [32]:
print("========== FINAL DATASET AUDIT ==========")

print("Total observations:", len(clinical_df))
print("Unique samples:", clinical_df["Sample_ID"].nunique())
print("Unique genes:", clinical_df["Hugo_Symbol"].nunique())

print("\nMissing values:")
print(clinical_df.isna().sum())

print("\nPending assessments:")
print(
    clinical_df["AMP_ASCO_CAP_Tier"]
    .astype(str)
    .str.contains("Pending", case=False)
    .sum()
)

print("\nTier distribution:")
print(clinical_df["AMP_ASCO_CAP_Tier"].value_counts())

print("\nObservations by gene:")
print(clinical_df["Hugo_Symbol"].value_counts())

print("\nUnique genomic alterations:")
print(
    clinical_df[
        ["Chromosome", "Start_Position",
         "Reference_Allele", "Tumor_Seq_Allele2"]
    ].drop_duplicates().shape[0]
)

print("\n========== AUDIT COMPLETE ==========")

========== FINAL DATASET AUDIT ==========
Total observations: 15
Unique samples: 5
Unique genes: 6

Missing values:
Sample_ID                           0
Hugo_Symbol                         0
Chromosome                          0
Start_Position                      0
Reference_Allele                    0
Tumor_Seq_Allele2                   0
HGVSc                               0
HGVSp_Short                         0
VEP_Consequence                     0
VEP_Impact                          0
gnomAD_AF                          12
COSMIC                              9
hotspot                             0
VAF                                 0
ClinVar_Evidence                    0
Oncology_Knowledgebase_Evidence     0
AMP_ASCO_CAP_Tier                   0
Therapeutic_Relevance               0
Evidence_Notes                      0
dtype: int64

Pending assessments:
0

Tier distribution:
AMP_ASCO_CAP_Tier
Tier III - provisional                                          7
Tier III - provisiona

In [33]:
# Standardize Tier III labels while preserving Tier I

tier_col = "AMP_ASCO_CAP_Tier"

tier3_mask = clinical_df[tier_col].astype(str).str.startswith("Tier III")

clinical_df.loc[tier3_mask, tier_col] = (
    "Tier III - provisional; uncertain clinical significance in LUAD"
)

# Update the exact EGFR L858R evidence
egfr_mask = (
    (clinical_df["Hugo_Symbol"] == "EGFR") &
    (clinical_df["HGVSc"] == "c.2573T>G") &
    (clinical_df["HGVSp_Short"] == "p.L858R")
)

clinical_df.loc[egfr_mask, "ClinVar_Evidence"] = (
    "ClinVar Variation ID 16609; RCV000435684.15. "
    "Somatic clinical impact Tier I - Strong for NSCLC; "
    "therapeutic sensitivity/response."
)

clinical_df.loc[egfr_mask, "AMP_ASCO_CAP_Tier"] = (
    "Tier I - Strong"
)

clinical_df.loc[egfr_mask, "Evidence_Notes"] = (
    "Exact EGFR c.2573T>G (p.Leu858Arg), GRCh38 chr7:55191822. "
    "ClinVar reports somatic Tier I therapeutic sensitivity/response "
    "for NSCLC (3 submissions). A separate ClinVar record reports "
    "a somatic pathogenic assertion for lung adenocarcinoma. "
    "Clinical interpretation remains dependent on tumour context."
)

# Save the standardized table
clinical_df.to_csv(file_path, index=False)

print("Clinical evidence table standardized.")
print("EGFR L858R records updated:", egfr_mask.sum())

print("\nStandardized tier distribution:")
print(clinical_df[tier_col].value_counts())

print("\nRemaining pending assessments:")
print(
    clinical_df[tier_col]
    .astype(str)
    .str.contains("Pending", case=False)
    .sum()
)

Clinical evidence table standardized.
EGFR L858R records updated: 1

Standardized tier distribution:
AMP_ASCO_CAP_Tier
Tier III - provisional; uncertain clinical significance in LUAD    14
Tier I - Strong                                                     1
Name: count, dtype: int64

Remaining pending assessments:
0


In [34]:
import os
import pandas as pd

base_dir = "/content/Task_3_Variant_Annotation"
table_dir = os.path.join(base_dir, "results", "tables")
os.makedirs(table_dir, exist_ok=True)

# 1. Gene-level summary
gene_summary = (
    clinical_df.groupby("Hugo_Symbol")
    .agg(
        Variant_Observations=("Hugo_Symbol", "size"),
        Samples=("Sample_ID", "nunique"),
        Unique_Genomic_Alterations=("Start_Position", "nunique"),
        Mean_VAF=("VAF", "mean"),
        VEP_Consequences=("VEP_Consequence",
                          lambda x: "; ".join(sorted(x.dropna().unique())))
    )
    .reset_index()
    .sort_values("Variant_Observations", ascending=False)
)

gene_summary["Mean_VAF"] = gene_summary["Mean_VAF"].round(4)

gene_summary.to_csv(
    os.path.join(table_dir, "LUAD_gene_level_summary.csv"),
    index=False
)

# 2. Clinical-tier summary
tier_summary = (
    clinical_df.groupby("AMP_ASCO_CAP_Tier")
    .agg(
        Variant_Observations=("Sample_ID", "size"),
        Unique_Samples=("Sample_ID", "nunique"),
        Genes=("Hugo_Symbol",
               lambda x: ", ".join(sorted(x.dropna().unique())))
    )
    .reset_index()
)

tier_summary.to_csv(
    os.path.join(table_dir, "LUAD_clinical_tier_summary.csv"),
    index=False
)

# 3. Variant consequence summary
consequence_summary = (
    clinical_df.groupby(["VEP_Consequence", "VEP_Impact"])
    .size()
    .reset_index(name="Variant_Observations")
    .sort_values("Variant_Observations", ascending=False)
)

consequence_summary.to_csv(
    os.path.join(table_dir, "LUAD_consequence_summary.csv"),
    index=False
)

print("Summary tables saved successfully.")

print("\n--- GENE-LEVEL SUMMARY ---")
display(gene_summary)

print("\n--- CLINICAL-TIER SUMMARY ---")
display(tier_summary)

print("\n--- CONSEQUENCE SUMMARY ---")
display(consequence_summary)

print("\nFiles saved in:", table_dir)

Summary tables saved successfully.

--- GENE-LEVEL SUMMARY ---


,Hugo_Symbol,Variant_Observations,Samples,Unique_Genomic_Alterations,Mean_VAF,VEP_Consequences
5,TP53,7,5,6,0.4086,missense_variant; splice_acceptor_variant
1,EGFR,3,2,3,0.2405,missense_variant
0,BRAF,2,2,2,0.0955,missense_variant
2,KEAP1,1,1,1,0.2884,missense_variant
3,NF1,1,1,1,0.2286,missense_variant
4,RB1,1,1,1,0.6957,missense_variant



--- CLINICAL-TIER SUMMARY ---


,AMP_ASCO_CAP_Tier,Variant_Observations,Unique_Samples,Genes
0,Tier I - Strong,1,1,EGFR
1,Tier III - provisional; uncertain clinical sig...,14,5,"BRAF, EGFR, KEAP1, NF1, RB1, TP53"



--- CONSEQUENCE SUMMARY ---


,VEP_Consequence,VEP_Impact,Variant_Observations
0,missense_variant,MODERATE,14
1,splice_acceptor_variant,HIGH,1



Files saved in: /content/Task_3_Variant_Annotation/results/tables


In [35]:
# Recalculate unique genomic alterations using the complete variant identity

gene_summary = (
    clinical_df.groupby("Hugo_Symbol")
    .agg(
        Variant_Observations=("Sample_ID", "size"),
        Samples=("Sample_ID", "nunique"),
        Unique_Genomic_Alterations=(
            "Start_Position",
            lambda x: clinical_df.loc[
                x.index,
                [
                    "Chromosome",
                    "Start_Position",
                    "Reference_Allele",
                    "Tumor_Seq_Allele2"
                ]
            ].drop_duplicates().shape[0]
        ),
        Mean_VAF=("VAF", "mean"),
        VEP_Consequences=(
            "VEP_Consequence",
            lambda x: "; ".join(sorted(x.dropna().unique()))
        )
    )
    .reset_index()
    .sort_values("Variant_Observations", ascending=False)
)

gene_summary["Mean_VAF"] = gene_summary["Mean_VAF"].round(4)

gene_summary.to_csv(
    os.path.join(
        table_dir,
        "LUAD_gene_level_summary.csv"
    ),
    index=False
)

print("Corrected gene-level summary:")
display(gene_summary)

Corrected gene-level summary:


,Hugo_Symbol,Variant_Observations,Samples,Unique_Genomic_Alterations,Mean_VAF,VEP_Consequences
5,TP53,7,5,6,0.4086,missense_variant; splice_acceptor_variant
1,EGFR,3,2,3,0.2405,missense_variant
0,BRAF,2,2,2,0.0955,missense_variant
2,KEAP1,1,1,1,0.2884,missense_variant
3,NF1,1,1,1,0.2286,missense_variant
4,RB1,1,1,1,0.6957,missense_variant


In [36]:
import sys
import platform
import pandas as pd
import requests
import json
import os
from datetime import date

metadata = {
    "Analysis": "TCGA-LUAD somatic variant annotation",
    "Data_Source": "NCI Genomic Data Commons (GDC)",
    "GDC_API": "https://api.gdc.cancer.gov",
    "GDC_Project": "TCGA-LUAD",
    "Data_Category": "Simple Nucleotide Variation",
    "Data_Type": "Masked Somatic Mutation",
    "Data_Format": "MAF",
    "Access_Level": "Open access",
    "Genome_Assembly": "GRCh38",
    "Samples_Analysed": 5,
    "Variant_Observations": 15,
    "Unique_Genomic_Alterations": 14,
    "Annotation_Tool": "Ensembl Variant Effect Predictor (VEP)",
    "VEP_Endpoint": "https://rest.ensembl.org/vep/human/region",
    "VEP_Parameters": "hgvs=1; canonical=1; mane=1; variant_class=1",
    "Analysis_Date": str(date.today()),
    "Python_Version": sys.version.split()[0],
    "Pandas_Version": pd.__version__,
    "Requests_Version": requests.__version__,
    "Operating_System": platform.platform()
}

metadata_df = pd.DataFrame(
    list(metadata.items()),
    columns=["Parameter", "Value"]
)

metadata_path = os.path.join(
    base_dir,
    "reports",
    "LUAD_analysis_metadata.csv"
)

os.makedirs(os.path.dirname(metadata_path), exist_ok=True)

metadata_df.to_csv(metadata_path, index=False)

display(metadata_df)

print("\nMetadata saved:", metadata_path)

,Parameter,Value
0,Analysis,TCGA-LUAD somatic variant annotation
1,Data_Source,NCI Genomic Data Commons (GDC)
2,GDC_API,https://api.gdc.cancer.gov
3,GDC_Project,TCGA-LUAD
4,Data_Category,Simple Nucleotide Variation
5,Data_Type,Masked Somatic Mutation
6,Data_Format,MAF
7,Access_Level,Open access
8,Genome_Assembly,GRCh38
9,Samples_Analysed,5



Metadata saved: /content/Task_3_Variant_Annotation/reports/LUAD_analysis_metadata.csv


In [38]:
# Record the version lookup limitation transparently

metadata_df.loc[
    metadata_df["Parameter"] == "VEP_Release",
    "Value"
] = "Not retrieved - Ensembl REST request timed out"

metadata_df.loc[
    metadata_df["Parameter"] == "Ensembl_Data_Version",
    "Value"
] = "Not retrieved - Ensembl REST request timed out"

metadata_df.to_csv(metadata_path, index=False)

print("Metadata saved successfully.")
print("VEP annotation results remain unchanged.")
print("Version lookup limitation documented.")

Metadata saved successfully.
VEP annotation results remain unchanged.
Version lookup limitation documented.


In [39]:
!pip -q install python-docx

import os
import pandas as pd
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT

# Project paths
base_dir = "/content/Task_3_Variant_Annotation"
report_dir = os.path.join(base_dir, "reports")
table_dir = os.path.join(base_dir, "results", "tables")

os.makedirs(report_dir, exist_ok=True)

# Load final outputs
clinical_df = pd.read_csv(
    os.path.join(table_dir, "LUAD_clinical_evidence_assessment.csv")
)

gene_summary = pd.read_csv(
    os.path.join(table_dir, "LUAD_gene_level_summary.csv")
)

tier_summary = pd.read_csv(
    os.path.join(table_dir, "LUAD_clinical_tier_summary.csv")
)

consequence_summary = pd.read_csv(
    os.path.join(table_dir, "LUAD_consequence_summary.csv")
)

metadata_df = pd.read_csv(
    os.path.join(report_dir, "LUAD_analysis_metadata.csv")
)

# Create document
doc = Document()

section = doc.sections[0]
section.top_margin = Inches(0.7)
section.bottom_margin = Inches(0.7)
section.left_margin = Inches(0.8)
section.right_margin = Inches(0.8)

styles = doc.styles
styles["Normal"].font.name = "Arial"
styles["Normal"].font.size = Pt(10)

# Title
title = doc.add_heading(
    "Somatic Variant Annotation and Clinical Evidence Assessment in TCGA-LUAD",
    0
)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
p.add_run("Genomic Data Analyst Assignment – Task 3").bold = True

doc.add_paragraph(
    "Analysis date: 11 November 2026"
)

# Abstract
doc.add_heading("Abstract", level=1)

doc.add_paragraph(
    "This study presents a focused somatic variant annotation and clinical "
    "evidence assessment using publicly available masked somatic mutation "
    "data from The Cancer Genome Atlas Lung Adenocarcinoma project (TCGA-LUAD). "
    "Five sample-level Mutation Annotation Format (MAF) files were retrieved "
    "from the National Cancer Institute Genomic Data Commons (GDC). A total "
    "of 9,684 mutation records were combined and screened to prioritize "
    "protein-altering variants in selected cancer-associated genes. Fifteen "
    "sample-level variant observations involving six genes were selected "
    "for further assessment. Variant consequences and HGVS annotations were "
    "evaluated using the Ensembl Variant Effect Predictor (VEP). Clinical "
    "evidence was reviewed using available ClinVar, COSMIC and oncology "
    "knowledgebase information. One observation was assigned Tier I – Strong "
    "and fourteen were provisionally classified as Tier III in the LUAD "
    "context. These results are a focused bioinformatics assessment and "
    "not a clinical diagnostic interpretation."
)

# Objectives
doc.add_heading("1. Objectives", level=1)

objectives = [
    "Retrieve public somatic mutation data from TCGA-LUAD.",
    "Prioritize potentially relevant protein-altering variants.",
    "Annotate variants using Ensembl VEP.",
    "Evaluate genomic consequences, HGVS nomenclature and available evidence.",
    "Assess clinical significance using the AMP/ASCO/CAP somatic framework.",
    "Generate reproducible datasets, summary tables and a documented report."
]

for item in objectives:
    doc.add_paragraph(item, style="List Bullet")

# Dataset
doc.add_heading("2. Dataset and Data Retrieval", level=1)

doc.add_paragraph(
    "Data were obtained from the NCI Genomic Data Commons API. The selected "
    "project was TCGA-LUAD (Lung Adenocarcinoma). Open-access files were "
    "queried using the Simple Nucleotide Variation data category, Masked "
    "Somatic Mutation data type and MAF format."
)

doc.add_paragraph(
    "Five sample-level MAF files were downloaded. These are processed "
    "somatic mutation files containing existing mutation annotations; "
    "they are not raw sequencing reads or unannotated VCF files."
)

doc.add_heading("2.1 Sample information", level=2)

sample_ids = sorted(clinical_df["Sample_ID"].dropna().unique())

for sample in sample_ids:
    doc.add_paragraph(sample, style="List Bullet")

doc.add_paragraph(
    "The source MAF files used GRCh38 genomic coordinates. The five files "
    "contained 9,684 mutation records in total."
)

# Methods
doc.add_heading("3. Methodology", level=1)

doc.add_heading("3.1 Variant prioritization", level=2)

doc.add_paragraph(
    "The combined MAF dataset was screened using protein-altering variant "
    "classes, including missense, nonsense, frameshift, splice-site, "
    "in-frame insertion/deletion and translation-start-site alterations. "
    "This screening identified 7,284 records in the selected consequence "
    "classes. A focused gene panel comprising EGFR, KRAS, TP53, STK11, "
    "KEAP1, BRAF, MET, ERBB2, PIK3CA, RB1 and NF1 was then considered. "
    "Fifteen candidate observations across six genes were retained for "
    "detailed annotation and clinical evidence review."
)

doc.add_heading("3.2 Ensembl VEP annotation", level=2)

doc.add_paragraph(
    "The selected variants were submitted to the Ensembl Variant Effect "
    "Predictor REST API using the human region endpoint. The annotation "
    "request included HGVS generation, canonical transcript selection, "
    "MANE annotation and variant class information. All 15 submitted "
    "observations returned VEP results."
)

doc.add_paragraph(
    "The VEP annotations included consequence terms, predicted impact, "
    "transcript information and HGVS descriptions. Fourteen observations "
    "were classified as missense_variant with MODERATE impact, while one "
    "TP53 variant was classified as splice_acceptor_variant with HIGH "
    "impact."
)

doc.add_heading("3.3 Transcript-level verification", level=2)

doc.add_paragraph(
    "A transcript-dependent annotation difference was identified for a "
    "BRAF variant. The MAF annotation used transcript ENST00000288602, "
    "whereas the canonical VEP output selected ENST00000646891. The "
    "original transcript was independently checked using VEP. Both "
    "transcript-specific annotations were retained in the integrated "
    "dataset rather than replacing one with the other."
)

doc.add_heading("3.4 Clinical evidence assessment", level=2)

doc.add_paragraph(
    "Clinical evidence was reviewed at the exact-variant level wherever "
    "possible. Available ClinVar information, COSMIC identifiers and "
    "hotspot annotations were considered alongside population frequency "
    "and sequencing evidence. The AMP/ASCO/CAP framework was used to "
    "organize somatic clinical significance into Tiers I–IV."
)

doc.add_paragraph(
    "VEP consequence impact is a predicted molecular consequence and is "
    "not equivalent to clinical pathogenicity. Similarly, a gene-level "
    "association with cancer or a hotspot designation does not by itself "
    "establish therapeutic actionability for an individual variant."
)

# Results
doc.add_heading("4. Results", level=1)

doc.add_heading("4.1 Overall results", level=2)

results = [
    f"Samples analysed: {clinical_df['Sample_ID'].nunique()}",
    f"Variant observations prioritized: {len(clinical_df)}",
    "Unique genomic alterations: 14",
    f"Genes represented: {clinical_df['Hugo_Symbol'].nunique()}",
    "No pending clinical evidence assessments remained at final audit."
]

for item in results:
    doc.add_paragraph(item, style="List Bullet")

doc.add_heading("4.2 Gene-level summary", level=2)

table = doc.add_table(rows=1, cols=5)
table.style = "Light Shading Accent 1"
table.alignment = WD_TABLE_ALIGNMENT.CENTER

headers = [
    "Gene",
    "Observations",
    "Samples",
    "Unique alterations",
    "Mean VAF"
]

for i, header in enumerate(headers):
    table.rows[0].cells[i].text = header

for _, row in gene_summary.iterrows():
    cells = table.add_row().cells
    cells[0].text = str(row["Hugo_Symbol"])
    cells[1].text = str(row["Variant_Observations"])
    cells[2].text = str(row["Samples"])
    cells[3].text = str(row["Unique_Genomic_Alterations"])
    cells[4].text = str(round(row["Mean_VAF"], 4))

doc.add_paragraph(
    "TP53 was the most frequently represented gene, with seven observations "
    "across five samples and six unique genomic alterations. EGFR had three "
    "observations across two samples. The TP53 p.R273L alteration occurred "
    "in two independent samples, explaining the difference between total "
    "observations and unique genomic alterations."
)

doc.add_heading("4.3 VEP consequence summary", level=2)

table = doc.add_table(rows=1, cols=3)
table.style = "Light Shading Accent 1"

for i, header in enumerate(
    ["Consequence", "VEP impact", "Observations"]
):
    table.rows[0].cells[i].text = header

for _, row in consequence_summary.iterrows():
    cells = table.add_row().cells
    cells[0].text = str(row["VEP_Consequence"])
    cells[1].text = str(row["VEP_Impact"])
    cells[2].text = str(row["Variant_Observations"])

doc.add_heading("4.4 Clinical tier summary", level=2)

table = doc.add_table(rows=1, cols=3)
table.style = "Light Shading Accent 1"

for i, header in enumerate(
    ["AMP/ASCO/CAP tier", "Observations", "Genes"]
):
    table.rows[0].cells[i].text = header

for _, row in tier_summary.iterrows():
    cells = table.add_row().cells
    cells[0].text = str(row["AMP_ASCO_CAP_Tier"])
    cells[1].text = str(row["Variant_Observations"])
    cells[2].text = str(row["Genes"])

doc.add_paragraph(
    "One EGFR p.L858R observation was classified as Tier I – Strong based "
    "on the reviewed somatic clinical evidence for NSCLC. Fourteen "
    "observations were assigned a provisional Tier III classification "
    "because sufficient LUAD-specific clinical evidence for a higher tier "
    "was not established in this assessment. These provisional assignments "
    "should not be interpreted as evidence that the variants are benign."
)

# Therapeutic relevance
doc.add_heading("5. Therapeutic Relevance", level=1)

doc.add_paragraph(
    "EGFR p.L858R is a clinically established activating EGFR alteration "
    "associated with sensitivity to EGFR-targeted tyrosine kinase "
    "inhibitors in relevant non-small-cell lung cancer settings. Its "
    "identification illustrates the potential relevance of somatic variant "
    "annotation to targeted therapy research. Treatment decisions, however, "
    "require full clinical assessment and current treatment guidelines."
)

doc.add_paragraph(
    "The TP53, BRAF, KEAP1, NF1 and RB1 observations may be relevant to "
    "tumour biology and pathway-level research. However, no variant-specific "
    "LUAD therapeutic recommendation was established for these observations "
    "in the present assessment. Gene-level biological relevance should "
    "not be confused with validated drug sensitivity."
)

# Limitations
doc.add_heading("6. Limitations", level=1)

limitations = [
    "Only five TCGA-LUAD samples were selected; results are not representative of the entire LUAD cohort.",
    "The source files were processed masked somatic MAF files, not raw sequencing data.",
    "No matched-normal analysis was performed in this project.",
    "The candidate gene panel and focused selection strategy may exclude other biologically relevant variants.",
    "Population frequency and COSMIC fields were unavailable for some records in the source MAF files; missing values were retained as missing.",
    "VAF is influenced by tumour purity, local copy number and sequencing characteristics and cannot independently establish clonality or pathogenicity.",
    "Transcript selection can produce different HGVS protein numbering; the BRAF transcript discrepancy was documented and verified.",
    "The Ensembl REST software/data version lookup timed out during metadata collection. The VEP release was therefore not independently recorded.",
    "Clinical knowledgebases and evidence classifications can change; the assessment is a research exercise, not a clinical-grade molecular report.",
    "Provisional Tier III classifications reflect insufficient evidence for a higher tier in the reviewed LUAD context and do not imply benignity."
]

for item in limitations:
    doc.add_paragraph(item, style="List Bullet")

# Reproducibility
doc.add_heading("7. Reproducibility and Output Files", level=1)

doc.add_paragraph(
    "The analysis used Python 3.13.18, pandas 2.2.3 and requests 2.32.4 "
    "in a Linux-based Google Colab environment. The GDC API and Ensembl "
    "VEP REST endpoint, along with the annotation parameters, are recorded "
    "in LUAD_analysis_metadata.csv."
)

doc.add_paragraph(
    "Key output files include the prioritized variant dataset, raw VEP "
    "response, VEP annotation summary, integrated annotation dataset, "
    "BRAF transcript verification, clinical evidence assessment and "
    "gene/consequence/tier summary tables."
)

# References
doc.add_heading("8. Data Sources and References", level=1)

references = [
    "National Cancer Institute Genomic Data Commons: https://portal.gdc.cancer.gov/",
    "GDC API: https://api.gdc.cancer.gov/",
    "Ensembl Variant Effect Predictor: https://www.ensembl.org/info/docs/tools/vep/index.html",
    "Ensembl VEP REST API: https://rest.ensembl.org/",
    "ClinVar: https://www.ncbi.nlm.nih.gov/clinvar/",
    "COSMIC: https://cancer.sanger.ac.uk/cosmic",
    "OncoKB: https://www.oncokb.org/",
    "Li MM et al. Standards and Guidelines for the Interpretation and Reporting of Sequence Variants in Cancer: A Joint Consensus Recommendation of AMP, ASCO, and CAP. Journal of Molecular Diagnostics. 2017;19(1):4–23."
]

for item in references:
    doc.add_paragraph(item, style="List Number")

# Closing statement
doc.add_heading("9. Conclusion", level=1)

doc.add_paragraph(
    "This project demonstrates a reproducible workflow for retrieving "
    "public cancer mutation data, prioritizing candidate variants, "
    "performing transcript-aware functional annotation and organizing "
    "clinical evidence. The analysis identified 15 prioritized "
    "sample-level observations representing 14 unique genomic alterations "
    "across six genes in five TCGA-LUAD samples. EGFR p.L858R was the "
    "principal observation with strong clinical significance in the "
    "reviewed NSCLC context, while the remaining variants require further "
    "functional and clinical evidence. The resulting datasets and "
    "documentation provide a foundation for further validation and "
    "research."
)

# Save report
report_path = os.path.join(
    report_dir,
    "Task_3_TCGA_LUAD_Variant_Annotation_Report.docx"
)

doc.save(report_path)

print("FINAL REPORT CREATED SUCCESSFULLY")
print(report_path)
print("Report size (KB):", round(os.path.getsize(report_path) / 1024, 2))

FINAL REPORT CREATED SUCCESSFULLY
/content/Task_3_Variant_Annotation/reports/Task_3_TCGA_LUAD_Variant_Annotation_Report.docx
Report size (KB): 40.08


In [40]:
import os

base_dir = "/content/Task_3_Variant_Annotation"

for root, dirs, files in os.walk(base_dir):
    level = root.replace(base_dir, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for filename in sorted(files):
        path = os.path.join(root, filename)
        size_kb = os.path.getsize(path) / 1024
        print(f"{indent}    {filename} ({size_kb:.1f} KB)")

print("\n--- REQUIRED FILE CHECK ---")

required_files = [
    "data/processed/LUAD_prioritised_variants.csv",
    "results/annotated/LUAD_VEP_annotation_summary.csv",
    "results/annotated/LUAD_VEP_raw_results.json",
    "results/annotated/LUAD_integrated_annotation.csv",
    "results/annotated/BRAF_transcript_verification.csv",
    "results/tables/LUAD_clinical_evidence_assessment.csv",
    "results/tables/LUAD_gene_level_summary.csv",
    "results/tables/LUAD_clinical_tier_summary.csv",
    "results/tables/LUAD_consequence_summary.csv",
    "reports/LUAD_analysis_metadata.csv",
    "reports/Task_3_TCGA_LUAD_Variant_Annotation_Report.docx"
]

for item in required_files:
    exists = os.path.exists(os.path.join(base_dir, item))
    print(("OK" if exists else "MISSING"), "-", item)

Task_3_Variant_Annotation/
    reports/
        LUAD_analysis_metadata.csv (0.7 KB)
        Task_3_TCGA_LUAD_Variant_Annotation_Report.docx (40.1 KB)
    scripts/
    data/
        processed/
            LUAD_prioritised_variants.csv (2.8 KB)
        raw/
            TCGA-05-4382.maf.gz (510.1 KB)
            TCGA-17-Z031.maf.gz (658.0 KB)
            TCGA-55-7994.maf.gz (477.9 KB)
            TCGA-55-8506.maf.gz (567.6 KB)
            TCGA-L9-A7SV.maf.gz (490.5 KB)
            data_mutations.txt (0.1 KB)
    results/
        tables/
            LUAD_clinical_evidence_assessment.csv (8.5 KB)
            LUAD_clinical_tier_summary.csv (0.2 KB)
            LUAD_consequence_summary.csv (0.1 KB)
            LUAD_gene_level_summary.csv (0.3 KB)
        annotated/
            BRAF_transcript_verification.csv (0.3 KB)
            LUAD_VEP_annotation_summary.csv (2.2 KB)
            LUAD_VEP_raw_results.json (336.1 KB)
            LUAD_integrated_annotation.csv (4.9 KB)

--- REQUIRED FILE CHEC

In [41]:
import pandas as pd
import os

variant_file = (
    "/content/Task_3_Variant_Annotation/"
    "data/processed/LUAD_prioritised_variants.csv"
)

variants = pd.read_csv(variant_file)

print("Dataset shape:", variants.shape)
print("\nAvailable columns:")
for column in variants.columns:
    print("-", column)

print("\nFirst three records:")
display(variants.head(3))

Dataset shape: (15, 21)

Available columns:
- Sample_ID
- Hugo_Symbol
- NCBI_Build
- Chromosome
- Start_Position
- End_Position
- Reference_Allele
- Tumor_Seq_Allele2
- Variant_Classification
- Variant_Type
- HGVSc
- HGVSp
- HGVSp_Short
- Transcript_ID
- gnomAD_AF
- COSMIC
- hotspot
- t_depth
- t_ref_count
- t_alt_count
- VAF

First three records:


,Sample_ID,Hugo_Symbol,NCBI_Build,Chromosome,Start_Position,End_Position,Reference_Allele,Tumor_Seq_Allele2,Variant_Classification,Variant_Type,...,HGVSp,HGVSp_Short,Transcript_ID,gnomAD_AF,COSMIC,hotspot,t_depth,t_ref_count,t_alt_count,VAF
0,TCGA-05-4382,EGFR,GRCh38,chr7,55152582,55152582,G,T,Missense_Mutation,SNP,...,p.Arg222Leu,p.R222L,ENST00000275493,NaN,NaN,N,520,430,90,0.173077
1,TCGA-05-4382,EGFR,GRCh38,chr7,55163734,55163734,G,C,Missense_Mutation,SNP,...,p.Glu545Gln,p.E545Q,ENST00000275493,NaN,NaN,N,223,206,17,0.076233
2,TCGA-05-4382,BRAF,GRCh38,chr7,140753296,140753296,C,A,Missense_Mutation,SNP,...,p.Leu653Phe,p.L653F,ENST00000288602,NaN,NaN,N,170,157,13,0.076471


In [42]:
from pathlib import Path
from textwrap import dedent

base_dir = Path("/content/Task_3_Variant_Annotation")
scripts_dir = base_dir / "scripts"
scripts_dir.mkdir(parents=True, exist_ok=True)

script = dedent("""
import pandas as pd
from pathlib import Path

BASE = Path(__file__).resolve().parents[1]

candidate_file = BASE / "data/processed/LUAD_prioritised_variants.csv"
clinical_file = BASE / "results/tables/LUAD_clinical_evidence_assessment.csv"
integrated_file = BASE / "results/annotated/LUAD_integrated_annotation.csv"

tables_dir = BASE / "results/tables"
tables_dir.mkdir(parents=True, exist_ok=True)

variants = pd.read_csv(candidate_file)
clinical = pd.read_csv(clinical_file)
integrated = pd.read_csv(integrated_file)

# 1. Gene-level summary
variant_key = [
    "Chromosome", "Start_Position",
    "Reference_Allele", "Tumor_Seq_Allele2"
]

gene_summary = (
    variants.groupby("Hugo_Symbol")
    .agg(
        Observations=("Sample_ID", "size"),
        Samples=("Sample_ID", "nunique"),
        Mean_VAF=("VAF", "mean")
    )
    .reset_index()
)

unique_variants = (
    variants.drop_duplicates(["Hugo_Symbol"] + variant_key)
    .groupby("Hugo_Symbol")
    .size()
    .rename("Unique_Genomic_Alterations")
    .reset_index()
)

gene_summary = gene_summary.merge(unique_variants, on="Hugo_Symbol")
gene_summary = gene_summary[
    ["Hugo_Symbol", "Observations", "Samples",
     "Unique_Genomic_Alterations", "Mean_VAF"]
]
gene_summary["Mean_VAF"] = gene_summary["Mean_VAF"].round(4)
gene_summary.to_csv(
    tables_dir / "LUAD_gene_level_summary.csv", index=False
)

# 2. VEP consequence summary
consequence_summary = (
    integrated.groupby(["VEP_Consequence", "VEP_Impact"])
    .size()
    .reset_index(name="Observations")
    .sort_values("Observations", ascending=False)
)

consequence_summary.to_csv(
    tables_dir / "LUAD_consequence_summary.csv", index=False
)

# 3. Clinical tier summary
tier_summary = (
    clinical.groupby("AMP_ASCO_CAP_Tier")
    .agg(
        Observations=("Sample_ID", "size"),
        Samples=("Sample_ID", "nunique")
    )
    .reset_index()
)

tier_genes = (
    clinical.groupby("AMP_ASCO_CAP_Tier")["Hugo_Symbol"]
    .apply(lambda x: ", ".join(sorted(set(x.dropna().astype(str)))))
    .reset_index(name="Genes")
)

tier_summary = tier_summary.merge(
    tier_genes, on="AMP_ASCO_CAP_Tier"
)

tier_summary.to_csv(
    tables_dir / "LUAD_clinical_tier_summary.csv", index=False
)

print("Summary tables regenerated successfully.")
print("\\\\nGene-level summary:")
print(gene_summary.to_string(index=False))
print("\\\\nVEP consequence summary:")
print(consequence_summary.to_string(index=False))
print("\\\\nClinical tier summary:")
print(tier_summary.to_string(index=False))
""").strip()

script_path = scripts_dir / "generate_summary_tables.py"
script_path.write_text(script)

print("Reproducibility script created:")
print(script_path)
print("File size:", round(script_path.stat().st_size / 1024, 2), "KB")

Reproducibility script created:
/content/Task_3_Variant_Annotation/scripts/generate_summary_tables.py
File size: 2.46 KB


In [43]:
from pathlib import Path
from textwrap import dedent

base_dir = Path("/content/Task_3_Variant_Annotation")

readme = dedent("""
# Task 3: Somatic Variant Annotation in TCGA-LUAD

## 1. Project overview
This project prioritises and annotates selected somatic variants from
five publicly available TCGA Lung Adenocarcinoma (TCGA-LUAD) samples.

## 2. Data source
- Repository: NCI Genomic Data Commons (GDC)
- API: https://api.gdc.cancer.gov
- Project: TCGA-LUAD
- Data category: Simple Nucleotide Variation
- Data type: Masked Somatic Mutation
- Data format: MAF
- Access: Open
- Genome assembly: GRCh38
- Samples: TCGA-05-4382, TCGA-17-Z031, TCGA-55-7994,
  TCGA-55-8506, TCGA-L9-A7SV

The downloaded MAF files are GDC masked somatic mutation files
containing existing variant annotations; they are not raw VCF files.

## 3. Analysis workflow
1. Download five open-access TCGA-LUAD MAF files using the GDC API.
2. Combine the sample-level mutation records.
3. Prioritise protein-altering variants in selected cancer-relevant genes.
4. Annotate selected variants using Ensembl Variant Effect Predictor
   (VEP) REST API.
5. Compare original MAF HGVS annotations with VEP transcript annotations.
6. Review available population-frequency, COSMIC, hotspot and
   clinical evidence fields.
7. Assess clinical significance using available ClinVar and
   oncology evidence, applying the AMP/ASCO/CAP somatic framework
   cautiously in the LUAD context.
8. Generate gene-level, consequence-level and clinical-tier summaries.

## 4. VEP annotation
- Endpoint: https://rest.ensembl.org/vep/human/region
- Parameters: hgvs=1, canonical=1, mane=1, variant_class=1
- Input assembly: GRCh38
- Annotation retrieval date: 11 November 2026

The Ensembl release/data version could not be retrieved because
the version-information requests timed out. No release number
has therefore been assumed.

## 5. Main outputs
- data/processed/LUAD_prioritised_variants.csv
- results/annotated/LUAD_VEP_annotation_summary.csv
- results/annotated/LUAD_VEP_raw_results.json
- results/annotated/LUAD_integrated_annotation.csv
- results/annotated/BRAF_transcript_verification.csv
- results/tables/LUAD_clinical_evidence_assessment.csv
- results/tables/LUAD_gene_level_summary.csv
- results/tables/LUAD_clinical_tier_summary.csv
- results/tables/LUAD_consequence_summary.csv
- reports/LUAD_analysis_metadata.csv
- reports/Task_3_TCGA_LUAD_Variant_Annotation_Report.docx

## 6. Reproducibility
The script scripts/generate_summary_tables.py regenerates the
gene-level, VEP consequence and clinical-tier summary CSV files
from the saved candidate, integrated annotation and clinical
evidence datasets.

Run from the project directory:

    python scripts/generate_summary_tables.py

The saved raw VEP JSON and metadata provide the annotation
record and analysis details. Re-running the live VEP annotation
requires access to the Ensembl REST service.

## 7. Important interpretation notes
- VEP consequence severity (HIGH/MODERATE) describes predicted
  molecular impact, not clinical pathogenicity.
- VAF is not, by itself, proof of clonality or pathogenicity.
- Missing population-frequency or COSMIC values are treated as
  unavailable, not as evidence of absence.
- Clinical tiers are interpreted in the specific LUAD context.
- A variant's therapeutic relevance must not be inferred solely
  from its gene or from evidence in an unrelated cancer type.
- This is a research annotation exercise, not a clinical
  diagnostic or treatment recommendation.

## 8. Software and environment
Analysis was performed in Google Colab using Python, pandas and
requests. Exact environment versions and data-retrieval details
are recorded in reports/LUAD_analysis_metadata.csv.
""").strip() + "\n"

readme_path = base_dir / "README.md"
readme_path.write_text(readme, encoding="utf-8")

print("README created:", readme_path)
print("Size:", round(readme_path.stat().st_size / 1024, 2), "KB")
print("\nREADME preview:")
print(readme[:1200])

README created: /content/Task_3_Variant_Annotation/README.md
Size: 3.52 KB

README preview:
# Task 3: Somatic Variant Annotation in TCGA-LUAD

## 1. Project overview
This project prioritises and annotates selected somatic variants from
five publicly available TCGA Lung Adenocarcinoma (TCGA-LUAD) samples.

## 2. Data source
- Repository: NCI Genomic Data Commons (GDC)
- API: https://api.gdc.cancer.gov
- Project: TCGA-LUAD
- Data category: Simple Nucleotide Variation
- Data type: Masked Somatic Mutation
- Data format: MAF
- Access: Open
- Genome assembly: GRCh38
- Samples: TCGA-05-4382, TCGA-17-Z031, TCGA-55-7994,
  TCGA-55-8506, TCGA-L9-A7SV

The downloaded MAF files are GDC masked somatic mutation files
containing existing variant annotations; they are not raw VCF files.

## 3. Analysis workflow
1. Download five open-access TCGA-LUAD MAF files using the GDC API.
2. Combine the sample-level mutation records.
3. Prioritise protein-altering variants in selected cancer-relevant genes.
4. A